# Colab experiment
Run from the top in Google Colab. Choose the required runtime first. If dependency installation asks for a runtime restart, restart and run from the top again. Set RESUME_RUN_ID only for an explicitly verified compatible checkpoint. Use RECOVER_RUN_ID to deliver staged results without rerunning compute. For a shared folder, add its shortcut in My Drive or set PROJECT_PATH_OVERRIDE to the actual mounted folder path. Put credentials in Colab Secrets; never paste them into this notebook.


In [1]:
import json, sys, subprocess, importlib.metadata, time, uuid
from pathlib import Path
CONFIG = json.loads('{"budget":{"burn_rate_cu_per_hour":5.4,"estimated_hours":0.65,"max_hours":4.0},"confirmed":true,"context":{"confidence_floor":0.001,"dataset_id":"rdd2020_datasprint_2_0","dataset_revision":"2026-11-23","evaluation_config_hash":"coco101_iou50_95_bestf1_t4speed","evaluator_id":"official_score_py","evaluator_revision":"starter_kit_v1","image_resolution":768,"iou_threshold":0.5,"max_dets_per_image":100,"n_samples":2150,"split_hash":"train14691_val2150_test4200","split_id":"official_train_val_test"},"dependency_check":"required","drive_folder_id":"1n8ddoVvmlpff1ScGLi_WPw-pPBe_AyC1","drive_relative_path":"DataSprint_2.0_Road_Damage","experiment":{"auto_batch_by_vram":true,"batch_a100":64,"batch_l4":32,"batch_t4":16,"close_mosaic":10,"conf_floor":0.001,"cos_lr":true,"epochs":50,"fallback_weights":"yolo26s.pt","imgsz":768,"iou_nms":0.6,"lowlight_luma_thresh":60.0,"lowlight_target_luma":115.0,"max_det":100,"model_weights":"yolo26m.pt","optimizer":"auto","patience":15,"robustness_aug_p":0.3,"seed":42,"smoke_only":false,"use_fp16_inference":true},"experiment_name":"datasprint2_full14691_yolo26m_768_eda_v3","gpu_required":true,"metric_context":{"ab_lowlight_delta_final_pts":{"higher_is_better":true,"split":"val_ab_test","unit":"score_delta_pts"},"accuracy_score_100":{"higher_is_better":true,"split":"val_clean","unit":"score_0_100"},"ap50_D00":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"ap50_D10":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"ap50_D20":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"ap50_D40":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"effective_batch_size":{"higher_is_better":true,"split":"train","unit":"count"},"estimated_compute_units":{"higher_is_better":false,"split":"runtime","unit":"CU"},"final_score_proxy_100":{"higher_is_better":true,"split":"val_combined_proxy","unit":"score_0_100"},"mAP50":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"mAP50_95":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"median_ms_t4":{"higher_is_better":false,"split":"val_clean","unit":"ms"},"peak_vram_gib":{"higher_is_better":false,"split":"runtime","unit":"GiB"},"precision":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"recall":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"robustness_mAP50_synth":{"higher_is_better":true,"split":"val_degraded_synth","unit":"ratio_0_1"},"speed_score_t4":{"higher_is_better":true,"split":"val_clean","unit":"ratio_0_1"},"train_images_count":{"higher_is_better":true,"split":"train","unit":"count"},"val_images_count":{"higher_is_better":true,"split":"val","unit":"count"},"winner_uses_lowlight_rescue":{"higher_is_better":true,"split":"val_ab_test","unit":"binary_flag"}},"min_free_disk_gib":20,"min_gpu_memory_gib":14,"packages":["ultralytics==8.4.216"],"project_id":"datasprint-2-0-road-damage","schema_version":"colab-builder-config/1","source_fingerprints":{"builder_sha256":"40efd2603d219b5ffce5bca7f5149ed20644fb0e49db68cc529b6b4f86a998a0","runtime_sha256":"e30c67b27342488b5304b3b938070e2fbe06c8a8cdcc3064cc645e732c2f906c","task_sha256":"ff42676531ad5b34171905d3c30b9245794447dc37663f346b0be2a3e9306549"},"workload":"training"}')
RESUME_RUN_ID = ''  # Set an existing run ID only when resuming.
RECOVER_RUN_ID = ''  # Deliver saved results only; does not rerun the task.
PROJECT_PATH_OVERRIDE = ''  # Optional path inside /content/drive.
START_NEW_RUN = False  # Set True for a fresh independent attempt.
if not RECOVER_RUN_ID and CONFIG.get('python_min') and sys.version_info[:2] < tuple(map(int, CONFIG['python_min'].split('.'))):
    raise RuntimeError('Python version below project requirement; choose a compatible Colab runtime')
if not RECOVER_RUN_ID and CONFIG.get('python_max_exclusive') and sys.version_info[:2] >= tuple(map(int, CONFIG['python_max_exclusive'].split('.'))):
    raise RuntimeError('Python version outside project compatibility range')
if RESUME_RUN_ID and RECOVER_RUN_ID:
    raise ValueError('Select recovery or checkpoint resumption, not both')
if RESUME_RUN_ID or RECOVER_RUN_ID:
    RUN_ID = RESUME_RUN_ID or RECOVER_RUN_ID
elif START_NEW_RUN or 'RUN_ID' not in globals() or globals().get('RUN_PROJECT_ID') != CONFIG['project_id']:
    RUN_ID = str(uuid.uuid4())
RUN_PROJECT_ID = CONFIG['project_id']
print('Run ID:', RUN_ID)
import importlib.util
RUNTIME_SOURCE = '"""Standard-library logging/recovery. No network calls or ML dependencies.\n\nAuditing third-party requirement markers uses the installed packaging parser.\nStandard-library-only EDA and event recovery do not require that package.\n\nLocal locks/CAS do not provide distributed locking on a mounted Google Drive.\nSerialize remote writers and reconcile destination markers before retrying writes.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport os\nimport re\nimport tempfile\nimport uuid\nimport importlib.metadata\nimport subprocess\nimport sys\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nSCHEMA_VERSION = 2\n_ID = re.compile(r"[A-Za-z0-9][A-Za-z0-9_-]{0,95}\\Z")\n_SECRET = re.compile(r"(?:^|_)(?:api_key|access_token|refresh_token|hf_token|password|secret|authorization|cookie|credentials|private_key|secret_access_key|access_key_id)$")\nDEFAULT_CONTEXT_KEYS = (\n    "dataset_id", "dataset_revision", "split_id", "split_hash",\n    "evaluator_id", "evaluator_revision", "evaluation_config_hash", "n_samples",\n)\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat(timespec="microseconds")\n\n\ndef check_json(value, path="root"):\n    """Reject non-JSON objects, non-finite values, and credential-bearing keys."""\n    if value is None or isinstance(value, (str, bool, int)):\n        return\n    if isinstance(value, float):\n        if not math.isfinite(value):\n            raise ValueError(f"Non-finite number at {path}")\n        return\n    if isinstance(value, list):\n        for i, child in enumerate(value):\n            check_json(child, f"{path}[{i}]")\n        return\n    if isinstance(value, dict):\n        for key, child in value.items():\n            if not isinstance(key, str):\n                raise TypeError("JSON object keys must be strings")\n            snake = re.sub(r"([a-z0-9])([A-Z])", r"\\1_\\2", key)\n            normalized = re.sub(r"[^a-z0-9]+", "_", snake.lower()).strip("_")\n            if _SECRET.search(normalized) or normalized in {"token", "apikey"}:\n                raise ValueError("Credential-bearing field rejected; use Colab Secrets")\n            check_json(child, f"{path}.{key}")\n        return\n    raise TypeError(f"Unsupported JSON value at {path}; convert explicitly")\n\n\ndef canonical(value):\n    check_json(value)\n    return json.dumps(value, sort_keys=True, ensure_ascii=False,\n                      separators=(",", ":"), allow_nan=False).encode("utf-8")\n\n\ndef digest(value):\n    return hashlib.sha256(canonical(value)).hexdigest()\n\n\ndef file_digest(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef strict_loads(text):\n    # Reject NaN/Infinity rather than silently accepting JavaScript extensions.\n    def reject(_):\n        raise ValueError("Invalid JSON numeric constant")\n    def unique_pairs(pairs):\n        result = {}\n        for key, value in pairs:\n            if key in result:\n                raise ValueError("Duplicate JSON object key")\n            result[key] = value\n        return result\n    def finite_float(raw):\n        number = float(raw)\n        if not math.isfinite(number):\n            raise ValueError("Invalid JSON numeric constant")\n        return number\n    return json.loads(text, parse_constant=reject, object_pairs_hook=unique_pairs, parse_float=finite_float)\n\n\ndef read_json(path):\n    return strict_loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef component(value):\n    if not isinstance(value, str) or not _ID.fullmatch(value):\n        raise ValueError("Use a nonempty, safe ID (letters, digits, hyphen, underscore)")\n    return value\n\n\ndef within(root, relative):\n    root = Path(root).resolve()\n    rel = Path(relative)\n    if rel.is_absolute() or ".." in rel.parts:\n        raise ValueError("Expected a project-relative path")\n    target = (root / rel).resolve()\n    if target == root or root not in target.parents:\n        raise ValueError("Path escapes project or names its root")\n    return target\n\n\ndef resolve_project_root(mount_root, relative_path, project_id, *, override=\'\', folder_id=None):\n    """Resolve an explicit mounted path and verify identity before ANY write.\n\n    Shared-folder visibility is not inferred from connector access. An override\n    must remain inside the mount and point at the verified project registry.\n    """\n    mount = Path(mount_root).resolve()\n    if override:\n        target = Path(override)\n        if not target.is_absolute():\n            target = mount / target\n        target = target.resolve()\n        if mount not in target.parents:\n            raise ValueError(\'Project override must be inside the mounted Drive\')\n    else:\n        target = within(mount / \'MyDrive\', relative_path)\n    state_path = target / \'05_reports/autopilot/project_state.json\'\n    if not state_path.is_file():\n        raise FileNotFoundError(\'Project is not visible in mounted Drive. Add its shortcut to My Drive, \'\n                                \'or set PROJECT_PATH_OVERRIDE to its mounted path. Mount an account \'\n                                \'that can access the folder. No replacement project was created.\')\n    state = read_json(state_path)\n    if state.get(\'schema_version\') != SCHEMA_VERSION or state.get(\'project_id\') != project_id:\n        raise ValueError(\'Mounted project identity/schema mismatch\')\n    if folder_id and state.get(\'registry\', {}).get(\'root_folder\', {}).get(\'id\') != folder_id:\n        raise ValueError(\'Mounted Drive folder ID mismatch\')\n    return target\n\n\ndef dependency_report(pins, *, policy=\'required\', run=subprocess.run,\n                      requires=importlib.metadata.requires):\n    """Inspect pip check, blocking workload dependency conflicts.\n\n    Unrelated preinstalled conflicts are surfaced separately; unknown checker\n    failures still stop execution. No environment-wide repair or uninstall.\n    """\n    if policy not in {\'required\', \'all\'}:\n        raise ValueError(\'Unknown dependency policy\')\n    normalize = lambda name: re.sub(r\'[-_.]+\', \'-\', name.lower())\n    roots = {normalize(pin.split(\'==\', 1)[0]) for pin in pins}\n    required = set(roots)\n    requested_extras = {name: set() for name in roots}\n    processed = {}\n    queue = list(required)\n    while queue:\n        name = queue.pop()\n        active_extras = requested_extras[name]\n        if name in processed and processed[name] == active_extras:\n            continue\n        processed[name] = set(active_extras)\n        try:\n            dependencies = requires(name) or []\n        except importlib.metadata.PackageNotFoundError:\n            if name in roots:\n                raise RuntimeError(\'A required package is missing\') from None\n            continue  # Optional/conditional requirement; pip check judges runtime applicability.\n        for requirement in dependencies:\n            try:\n                from packaging.requirements import Requirement\n            except ImportError:\n                raise RuntimeError(\'Requirement metadata needs the packaging parser; include a compatible packaging pin\') from None\n            try:\n                parsed = Requirement(requirement)\n                if parsed.marker and not any(parsed.marker.evaluate({\'extra\': extra})\n                                             for extra in {\'\'} | active_extras):\n                    continue\n            except (ValueError, KeyError):\n                raise RuntimeError(\'Cannot interpret a required dependency marker\') from None\n            child = normalize(parsed.name)\n            extras = {normalize(extra) for extra in parsed.extras}\n            if child not in requested_extras or not extras <= requested_extras[child]:\n                requested_extras.setdefault(child, set()).update(extras)\n                required.add(child)\n                queue.append(child)\n    checked = run([sys.executable, \'-m\', \'pip\', \'check\'], capture_output=True, text=True)\n    if checked.returncode == 0:\n        return {\'status\': \'clean\', \'unrelated_conflicts\': 0, \'required_packages\': sorted(required)}\n    conflicts = []\n    for line in checked.stdout.splitlines():\n        match = re.match(r\'^([A-Za-z0-9_.-]+)\\s+\\S+\\s+(?:requires|has requirement)\\s+([A-Za-z0-9_.-]+)\', line)\n        if not match:\n            raise RuntimeError(\'Dependency checker failed or returned an unrecognized conflict\')\n        conflicts.append((normalize(match[1]), normalize(match[2])))\n    if not conflicts or checked.stderr.strip():\n        raise RuntimeError(\'Dependency checker did not produce a valid report\')\n    # Only requirements OF the workload closure constrain this run. An unrelated\n    # reverse dependent (for example an unused app constraining numpy) does not.\n    if policy == \'all\' or any(a in required for a, b in conflicts):\n        raise RuntimeError(\'Required dependency conflict; diagnose before running this workload\')\n    return {\'status\': \'unrelated_conflicts\', \'unrelated_conflicts\': len(conflicts),\n            \'required_packages\': sorted(required)}\n\n\n@contextmanager\ndef local_lock(path):\n    """Fail closed on contention. Never auto-delete another writer\'s lock."""\n    lock = Path(str(path) + ".lock")\n    lock.parent.mkdir(parents=True, exist_ok=True)\n    fd = os.open(str(lock), os.O_CREAT | os.O_EXCL | os.O_WRONLY, 0o600)\n    try:\n        os.write(fd, str(os.getpid()).encode("ascii"))\n        os.close(fd)\n        fd = None\n        yield\n    finally:\n        if fd is not None:\n            os.close(fd)\n        lock.unlink()\n\n\ndef atomic_json(path, value, *, expected=None, absent=False):\n    """Atomic same-directory replace plus a local cooperative lock and SHA CAS."""\n    path = Path(path)\n    payload = canonical(value)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with local_lock(path):\n        if absent and path.exists():\n            raise FileExistsError(str(path))\n        if expected is not None and (not path.exists() or file_digest(path) != expected):\n            raise RuntimeError("State changed; reload and reconcile")\n        fd, temporary = tempfile.mkstemp(prefix=".pending-", dir=path.parent)\n        try:\n            with os.fdopen(fd, "wb") as out:\n                out.write(payload)\n                out.flush()\n                os.fsync(out.fileno())\n            os.replace(temporary, path)\n        finally:\n            if os.path.exists(temporary):\n                os.unlink(temporary)\n    # Readback checks local bytes only; it cannot prove Drive server durability.\n    if path.read_bytes() != payload:\n        raise IOError("JSON write readback mismatch")\n    return path\n\n\ndef _numeric(value):\n    return (not isinstance(value, bool) and\n            (isinstance(value, int) or isinstance(value, float) and math.isfinite(value)))\n\n\ndef validate_metrics(metrics):\n    if not isinstance(metrics, dict):\n        raise TypeError("metrics must be a JSON object")\n    for name, value in metrics.items():\n        if not isinstance(name, str) or not name or not _numeric(value):\n            raise ValueError("Metric names must be nonempty strings and values finite numbers")\n\n\nclass LogDeliveryError(OSError):\n    """The log is staged locally but its project destination was not verified."""\n    def __init__(self, spool_path, error_category):\n        self.spool_path = Path(spool_path)\n        self.error_category = error_category\n        super().__init__(f"Log delivery pending ({error_category}); recover from {self.spool_path}")\n\n\ndef _write_event(path, record):\n    try:\n        atomic_json(path, record, absent=True)\n    except FileExistsError:\n        if not path.is_file():\n            raise\n        prior = read_json(path)\n        body = {k: v for k, v in prior.items() if k not in {"timestamp_utc", "payload_digest"}}\n        expected = record["payload_digest"]\n        if prior.get("payload_digest") != expected or digest(body) != expected:\n            raise FileExistsError("Event ID already contains different data; choose a revision ID") from None\n    return path\n\n\ndef validate_context(context, metric_context):\n    if not isinstance(context,dict) or not isinstance(metric_context,dict):\n        raise ValueError(\'Evaluation context and metric_context must be objects\')\n    if any(not isinstance(key,str) or not key.strip() or not isinstance(value,dict)\n           for key,value in metric_context.items()):\n        raise ValueError(\'Each metric context must be a named object\')\n    check_json(context)\n    check_json(metric_context)\n\n\ndef validate_event(record, *, run_id=None, project_id=None):\n    """Validate a saved event before sync or recovery; never execute its source."""\n    check_json(record)\n    if not isinstance(record, dict) or record.get(\'schema_version\') != SCHEMA_VERSION:\n        raise ValueError(\'Unsupported event schema\')\n    component(record.get(\'run_id\'))\n    component(record.get(\'event_id\'))\n    if run_id and record[\'run_id\'] != run_id:\n        raise ValueError(\'Event run identity mismatch\')\n    if not isinstance(record.get(\'config\'), dict):\n        raise ValueError(\'Event configuration is missing\')\n    if project_id and record[\'config\'].get(\'project_id\') != project_id:\n        raise ValueError(\'Event project identity mismatch\')\n    if record.get(\'status\') not in {\'completed\', \'partial\', \'failed\', \'interrupted\'}:\n        raise ValueError(\'Invalid event status\')\n    validate_context(record.get(\'context\', {}), record.get(\'metric_context\', {}))\n    validate_metrics(record.get(\'metrics\'))\n    if record[\'status\'] == \'completed\' and not record[\'metrics\']:\n        raise ValueError(\'Completed event lacks metrics\')\n    body = {k: v for k, v in record.items() if k not in {\'timestamp_utc\', \'payload_digest\'}}\n    if digest(body) != record.get(\'payload_digest\'):\n        raise ValueError(\'Event payload checksum mismatch\')\n    return record\n\n\ndef copy_immutable_file(source, destination, expected_sha256, *, max_bytes=16 * 1024 * 1024):\n    """Recover bounded task artifacts; refuse oversized files and replacements."""\n    source, destination = Path(source), Path(destination)\n    if source.stat().st_size > max_bytes:\n        raise ValueError(\'Recovery artifact is too large or has a mismatched checksum\')\n    with source.open(\'rb\') as stream:\n        payload = stream.read(max_bytes + 1)\n    if len(payload) > max_bytes or hashlib.sha256(payload).hexdigest() != expected_sha256:\n        raise ValueError(\'Recovery artifact is too large or has a mismatched checksum\')\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with local_lock(destination):\n        if destination.exists():\n            if file_digest(destination) != expected_sha256:\n                raise FileExistsError(\'Recovery destination contains different content\')\n            return destination\n        fd, temporary = tempfile.mkstemp(prefix=\'.recovery-\', dir=destination.parent)\n        try:\n            with os.fdopen(fd, \'wb\') as stream:\n                stream.write(payload)\n                stream.flush()\n                os.fsync(stream.fileno())\n            os.replace(temporary, destination)\n        finally:\n            if os.path.exists(temporary):\n                os.unlink(temporary)\n    if file_digest(destination) != expected_sha256:\n        raise IOError(\'Recovery artifact readback mismatch\')\n    return destination\n\n\ndef recover_spooled_events(project_root, spool_root, run_id, *, project_id):\n    """Deliver original staged events and bounded artifacts WITHOUT recompute."""\n    run_id = component(run_id)\n    state = read_json(within(project_root, \'05_reports/autopilot/project_state.json\'))\n    if state.get(\'schema_version\') != SCHEMA_VERSION or state.get(\'project_id\') != project_id:\n        raise ValueError(\'Recovery project mismatch\')\n    events = within(spool_root, f\'03_results/runs/{run_id}/events\')\n    delivered = []\n    for source in sorted(events.glob(\'*.json\')):\n        source = within(spool_root, f\'03_results/runs/{run_id}/events/{source.name}\')\n        record = validate_event(read_json(source), run_id=run_id, project_id=project_id)\n        if source.stem != record[\'event_id\']:\n            raise ValueError(\'Event path identity mismatch\')\n        artifacts = record.get(\'artifacts\', {}).get(\'files\', {})\n        if not isinstance(artifacts, dict):\n            raise ValueError(\'Invalid artifact manifest\')\n        for relative, sha256 in artifacts.items():\n            copy_immutable_file(within(spool_root, relative), within(project_root, relative), sha256)\n        destination = within(project_root, f\'03_results/runs/{run_id}/events/{source.name}\')\n        _write_event(destination, record)\n        observed = validate_event(read_json(destination), run_id=run_id, project_id=project_id)\n        if observed[\'payload_digest\'] != record[\'payload_digest\']:\n            raise IOError(\'Recovered event readback mismatch\')\n        delivered.append({\'event_id\': record[\'event_id\'], \'status\': record[\'status\']})\n    return delivered\n\n\ndef log_experiment(project_root, exp_id, metrics, *, run_id=None, event_id="final",\n                   status="completed", model_name=None, config=None, context=None,\n                   metric_context=None, artifacts=None, notes="", errors=None,\n                   spool_root=None):\n    """Write an immutable run event. Retry with the SAME run_id/event_id.\n\n    A different payload at an existing ID raises; log corrections under a new\n    event_id. Returned files contain JSON primitives only. No secret values print.\n    """\n    run_id = component(run_id or str(uuid.uuid4()))\n    event_id = component(event_id)\n    if status not in {"completed", "partial", "failed", "interrupted"}:\n        raise ValueError("Unknown run status")\n    if config is not None and not isinstance(config,dict):\n        raise ValueError("Experiment config must be an object")\n    validate_context({} if context is None else context, {} if metric_context is None else metric_context)\n    validate_metrics(metrics)\n    if status == "completed" and not metrics:\n        raise ValueError("Completed runs need observed metrics")\n    if artifacts is not None and not isinstance(artifacts, dict):\n        raise ValueError(\'Artifacts must be an object\')\n    files = (artifacts or {}).get(\'files\', {})\n    if not isinstance(files, dict) or any(not isinstance(path, str) or not isinstance(sha, str)\n            or not re.fullmatch(\'[0-9a-f]{64}\', sha) for path, sha in files.items()):\n        raise ValueError(\'Artifact files need relative paths and SHA-256 strings\')\n    # Invalid local exports are task/export failures, not completed results to\n    # recover. Validate before staging an immutable terminal event. Delivery\n    # still copies/checks again to detect a subsequent source change.\n    if spool_root is not None:\n        for relative, expected_hash in files.items():\n            source = within(spool_root, relative)\n            if not source.is_file() or source.stat().st_size > 16 * 1024 * 1024 or file_digest(source) != expected_hash:\n                raise ValueError(\'Staged task artifact is missing, oversized, or has a mismatched checksum\')\n    body = {\n        "schema_version": SCHEMA_VERSION, "run_id": run_id, "event_id": event_id,\n        "exp_id": exp_id, "status": status, "model_name": model_name,\n        "metrics": metrics, "config": config or {}, "context": context or {},\n        "metric_context": metric_context or {}, "artifacts": artifacts or {},\n        "notes": notes, "errors": errors or [],\n    }\n    payload_digest = digest(body)\n    path = within(project_root, f"03_results/runs/{run_id}/events/{event_id}.json")\n    record = {**body, "timestamp_utc": utc_now(), "payload_digest": payload_digest}\n    # Reuse the immutable destination timestamp when the same event is restaged\n    # into a new spool. A mount/read failure must not prevent local staging.\n    try:\n        previous = validate_event(read_json(path), run_id=run_id)\n    except (OSError, ValueError, TypeError):\n        previous = None\n    if previous is not None and previous[\'payload_digest\'] == payload_digest:\n        record = previous\n    spool_path = None\n    if spool_root is not None:\n        if Path(spool_root).resolve() == Path(project_root).resolve():\n            raise ValueError("Use separate local spool storage")\n        spool_path = within(spool_root, f"03_results/runs/{run_id}/events/{event_id}.json")\n        _write_event(spool_path, record)\n        # Deliver the original staged bytes/UTC time after an uncertain retry.\n        record = read_json(spool_path)\n    try:\n        for relative, expected_hash in record[\'artifacts\'].get(\'files\', {}).items():\n            destination = within(project_root, relative)\n            if spool_root is not None:\n                copy_immutable_file(within(spool_root, relative), destination, expected_hash)\n            elif not destination.is_file() or file_digest(destination) != expected_hash:\n                raise ValueError(\'Task artifact could not be verified\')\n        return _write_event(path, record)\n    except (OSError, ValueError) as error:\n        if spool_path is not None:\n            raise LogDeliveryError(spool_path, type(error).__name__) from None\n        raise\n\n\ndef save_checkpoint_manifest(project_root, run_id, checkpoint_id, config, files):\n    """Call only AFTER model/optimizer/RNG artifacts have been fully written."""\n    run_id, checkpoint_id = component(run_id), component(checkpoint_id)\n    if not files or not isinstance(files, list):\n        raise ValueError("List at least one finalized checkpoint artifact")\n    hashes = {}\n    for relative in files:\n        target = within(project_root, relative)\n        if not target.is_file():\n            raise FileNotFoundError(str(target))\n        hashes[Path(relative).as_posix()] = file_digest(target)\n    payload = {"schema_version": SCHEMA_VERSION, "run_id": run_id,\n               "checkpoint_id": checkpoint_id, "config_digest": digest(config),\n               "files": hashes, "complete": True}\n    path = within(project_root, f"03_results/runs/{run_id}/checkpoints/{checkpoint_id}.json")\n    try:\n        return atomic_json(path, payload, absent=True)\n    except FileExistsError:\n        if read_json(path) != payload:\n            raise FileExistsError("Checkpoint manifest is immutable") from None\n        return path\n\n\ndef verify_checkpoint(project_root, manifest_path, config, *, run_id):\n    manifest = read_json(manifest_path)\n    if (manifest.get("schema_version") != SCHEMA_VERSION or manifest.get("complete") is not True\n            or manifest.get("run_id") != component(run_id)\n            or manifest.get("config_digest") != digest(config) or not manifest.get("files")):\n        raise ValueError("Checkpoint context/run mismatch or incomplete manifest")\n    for relative, expected in manifest["files"].items():\n        if file_digest(within(project_root, relative)) != expected:\n            raise ValueError("Checkpoint artifact checksum mismatch")\n    return manifest\n\n\ndef seed_everything(seed=42, *, deterministic=True):\n    """Lock Python and any already-loaded NumPy/PyTorch RNGs; return JSON receipt."""\n    if isinstance(seed, bool) or not isinstance(seed, int) or seed < 0:\n        raise ValueError("seed must be a non-negative integer")\n    import random\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    random.seed(seed)\n    seeded = ["python_random"]\n    np_mod = sys.modules.get("numpy")\n    if np_mod is not None and hasattr(np_mod, "random"):\n        np_mod.random.seed(seed)\n        seeded.append("numpy")\n    torch_mod = sys.modules.get("torch")\n    if torch_mod is not None and hasattr(torch_mod, "manual_seed"):\n        torch_mod.manual_seed(seed)\n        seeded.append("torch_cpu")\n        if hasattr(torch_mod, "cuda") and torch_mod.cuda.is_available():\n            torch_mod.cuda.manual_seed_all(seed)\n            seeded.append("torch_cuda")\n        if hasattr(torch_mod, "backends") and hasattr(torch_mod.backends, "cudnn"):\n            torch_mod.backends.cudnn.deterministic = bool(deterministic)\n            torch_mod.backends.cudnn.benchmark = not bool(deterministic)\n    return {"seed": seed, "deterministic": bool(deterministic), "seeded_modules": seeded}\n\n\ndef stage_to_local_cache(project_root, relative_path, cache_root="/content/autopilot_cache", *, expected_sha256=None):\n    """Stage a Drive file to fast local Colab disk with SHA-256 verification."""\n    source = within(project_root, relative_path)\n    if not source.is_file():\n        raise FileNotFoundError(str(source))\n    target = within(cache_root, relative_path)\n    source_size = source.stat().st_size\n    if target.is_file() and target.stat().st_size == source_size:\n        cached_sha = file_digest(target)\n        if expected_sha256 is None or cached_sha == expected_sha256:\n            if expected_sha256 is not None or cached_sha == file_digest(source):\n                return {"local_path": str(target), "sha256": cached_sha,\n                        "size_bytes": source_size, "cache_hit": True}\n    source_sha = file_digest(source)\n    if expected_sha256 is not None and source_sha != expected_sha256:\n        raise ValueError("Source file SHA-256 mismatch before local caching")\n    target.parent.mkdir(parents=True, exist_ok=True)\n    fd, temporary = tempfile.mkstemp(prefix=".cache-", dir=target.parent)\n    try:\n        with os.fdopen(fd, "wb") as out, source.open("rb") as inp:\n            for block in iter(lambda: inp.read(1024 * 1024), b""):\n                out.write(block)\n            out.flush()\n            os.fsync(out.fileno())\n        os.replace(temporary, target)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n    if file_digest(target) != source_sha:\n        raise IOError("Local cache readback checksum mismatch")\n    return {"local_path": str(target), "sha256": source_sha,\n            "size_bytes": source_size, "cache_hit": False}\n\n\ndef collect_colab_telemetry(elapsed_seconds, *, burn_rate_cu_per_hour=None):\n    """Collect JSON-safe Colab compute telemetry (elapsed, peak VRAM, estimated CUs)."""\n    if isinstance(elapsed_seconds, bool) or not _numeric(elapsed_seconds) or elapsed_seconds < 0:\n        raise ValueError("elapsed_seconds must be a non-negative finite number")\n    telemetry = {"elapsed_seconds": float(elapsed_seconds)}\n    if burn_rate_cu_per_hour is not None:\n        if isinstance(burn_rate_cu_per_hour, bool) or not _numeric(burn_rate_cu_per_hour) or burn_rate_cu_per_hour < 0:\n            raise ValueError("burn_rate_cu_per_hour must be a non-negative finite number")\n        telemetry["burn_rate_cu_per_hour"] = float(burn_rate_cu_per_hour)\n        telemetry["estimated_compute_units"] = round((float(elapsed_seconds) / 3600.0) * float(burn_rate_cu_per_hour), 4)\n    torch_mod = sys.modules.get("torch")\n    if torch_mod is not None and hasattr(torch_mod, "cuda") and torch_mod.cuda.is_available():\n        try:\n            telemetry["peak_vram_gib"] = round(float(torch_mod.cuda.max_memory_allocated()) / (1024 ** 3), 4)\n        except Exception:\n            pass\n    return telemetry\n\n\nPROJECT_MEMORY_SCHEMA = "ml-project-memory/1"\nPROJECT_MEMORY_RELPATH = "05_reports/autopilot/project_memory.json"\n_MEMORY_SECTIONS = (\n    "setup_context", "destinations", "active_lineage",\n    "experiment_memory", "publication_memory", "next_experiments_queue",\n)\n\nDOMAIN_CATALOG = {\n    "classical": {\n        "display_name": "Classical Tabular ML",\n        "terminology": {"model_unit": "Estimator / Pipeline", "data_unit": "Feature matrix (rows/cols)", "split_strategy": "Stratified / Group K-Fold", "tuning_knobs": "n_estimators, max_depth, learning_rate, regularization, class_weight, decision_threshold"},\n        "tracked_parameters": ["estimator", "n_estimators", "max_depth", "learning_rate", "subsample", "colsample_bytree", "reg_alpha", "reg_lambda", "class_weight", "decision_threshold", "cv_folds"],\n        "candidate_metrics": ["roc_auc", "pr_auc", "f1_macro", "log_loss", "rmse", "mae", "r2", "brier_score"],\n        "extra_context_keys": ["cv_folds", "Positive_class_or_averaging", "decision_threshold"],\n        "leakage_and_split_controls": ["Fit imputers, scalers, and target encoders strictly on training folds", "Group split by entity_id when multiple rows share an entity"],\n        "sheets_views": ["Ablations & Hyperparameters", "EDA Findings"],\n        "docs_focus": "Feature engineering pipeline, out-of-fold validation vs held-out test, threshold calibration, and feature importance / SHAP ablations.",\n    },\n    "unsupervised": {\n        "display_name": "Unsupervised & Representation Learning",\n        "terminology": {"model_unit": "Clustering / Projection / Anomaly Detector", "data_unit": "Unlabeled feature vectors", "split_strategy": "Stability / Cohort Bootstrap", "tuning_knobs": "n_clusters, eps, min_samples, n_components, contamination, distance_metric"},\n        "tracked_parameters": ["algorithm", "n_clusters_or_components", "distance_metric", "contamination", "scaling_method"],\n        "candidate_metrics": ["silhouette_score", "davies_bouldin_index", "calinski_harabasz", "reconstruction_error", "cluster_stability_ari"],\n        "extra_context_keys": ["distance_metric", "feature_scaling"],\n        "leakage_and_split_controls": ["Do not fabricate supervised accuracy without ground-truth labels", "Evaluate stability across bootstrap resamples or temporal cohorts"],\n        "sheets_views": ["EDA Findings", "Ablations & Hyperparameters"],\n        "docs_focus": "Cluster/anomaly stability, representation quality, distance metric sensitivity, and domain sanity checks.",\n    },\n    "deep_learning": {\n        "display_name": "Deep Learning & Neural Architectures",\n        "terminology": {"model_unit": "Neural Backbone / Head", "data_unit": "Mini-batch tensors", "split_strategy": "Train / Val / Held-out Test", "tuning_knobs": "architecture, optimizer, lr_schedule, warmup_steps, batch_size, weight_decay, dropout, mixed_precision, grad_clip"},\n        "tracked_parameters": ["architecture", "optimizer", "learning_rate", "lr_scheduler", "warmup_ratio", "batch_size", "epochs_or_steps", "weight_decay", "dropout", "precision_dtype", "grad_accum_steps"],\n        "candidate_metrics": ["val_loss", "train_loss", "top1_accuracy", "f1_score", "throughput_samples_per_sec", "peak_vram_gib"],\n        "extra_context_keys": ["checkpoint_selection_metric", "precision_dtype"],\n        "leakage_and_split_controls": ["Select best checkpoint on validation split only, never held-out test", "Verify complete optimizer/scheduler/scaler/RNG state before claiming training resume"],\n        "sheets_views": ["Training Curves", "Ablations & Hyperparameters", "Colab Compute & Resource Ledger"],\n        "docs_focus": "Convergence curves, train-val generalization gap, architecture & regularization ablations, and checkpoint resume verification.",\n    },\n    "computer_vision": {\n        "display_name": "Computer Vision (Classification, Detection, Segmentation)",\n        "terminology": {"model_unit": "Vision Backbone / Detector / Segmenter", "data_unit": "Images / Video frames / Bounding boxes / Masks", "split_strategy": "Scene / Patient / Camera Group Split", "tuning_knobs": "backbone, image_resolution, augmentation_policy, iou_threshold, conf_threshold, anchor_or_query_config, focal_loss_gamma"},\n        "tracked_parameters": ["backbone", "pretrained_weights", "image_size", "augmentation_policy", "iou_threshold", "conf_threshold", "nms_type", "batch_size", "learning_rate", "precision_dtype"],\n        "candidate_metrics": ["mAP_50", "mAP_50_95", "mean_iou", "dice_score", "top1_accuracy", "fps_throughput", "latency_p95_ms"],\n        "extra_context_keys": ["image_resolution", "iou_threshold", "eval_augmentation_tta"],\n        "leakage_and_split_controls": ["Group split by video/patient/scene/camera to prevent near-duplicate frame leakage", "Apply random augmentations (MixUp, CutMix, flips) to training split only", "Stage image archives to /content/autopilot_cache to avoid Google Drive FUSE I/O bottlenecks"],\n        "sheets_views": ["Ablations & Hyperparameters", "Training Curves", "Colab Compute & Resource Ledger"],\n        "docs_focus": "Resolution & augmentation ablations, mAP/IoU across object sizes/classes, qualitative failure modes (occlusion, small objects), and FPS/latency tradeoff.",\n    },\n    "time_series": {\n        "display_name": "Time Series Forecasting & Sequential Modeling",\n        "terminology": {"model_unit": "Forecaster / Sequential Architecture", "data_unit": "Temporal series (entity x timestamp)", "split_strategy": "Rolling / Expanding Walk-Forward Window with Embargo", "tuning_knobs": "forecast_horizon, lookback_window, seasonality_periods, lag_features, exogenous_vars, scaling_per_series"},\n        "tracked_parameters": ["model_type", "forecast_horizon", "lookback_window", "frequency", "lag_features", "rolling_aggregations", "exogenous_features", "target_transform", "cv_windows"],\n        "candidate_metrics": ["mase", "smape", "wape", "rmse", "mae", "quantile_pinball_loss", "directional_accuracy"],\n        "extra_context_keys": ["forecast_horizon", "lookback_window", "temporal_cutoff_dates", "target_scale_space"],\n        "leakage_and_split_controls": ["Strict causal ordering: never use random K-Fold on time series", "Enforce embargo/gap between train end and validation forecast origin", "Compute rolling stats and scalers strictly using past window data (no future lookahead)", "Always compare against a Seasonal Naive / Last-Value baseline"],\n        "sheets_views": ["Ablations & Hyperparameters", "Dataset Splits", "EDA Findings"],\n        "docs_focus": "Walk-forward validation windows, Seasonal Naive baseline comparison, horizon decay analysis, lag/exogenous feature ablations, and regime-shift errors.",\n    },\n    "nlp": {\n        "display_name": "Natural Language Processing (Classification, NER, Sequence Tasks)",\n        "terminology": {"model_unit": "Language Encoder / Sequence Tagger", "data_unit": "Documents / Sentences / Token spans", "split_strategy": "Document / Author / Source-Deduplicated Split", "tuning_knobs": "pretrained_checkpoint, tokenizer, max_seq_length, pooling_strategy, learning_rate, warmup_ratio, label_smoothing"},\n        "tracked_parameters": ["pretrained_model", "tokenizer_revision", "max_seq_length", "learning_rate", "batch_size", "warmup_ratio", "weight_decay", "aggregation_level"],\n        "candidate_metrics": ["f1_macro", "f1_micro", "exact_match", "span_f1", "matthews_corrcoef", "roc_auc"],\n        "extra_context_keys": ["tokenizer_revision", "max_seq_length", "span_or_token_granularity"],\n        "leakage_and_split_controls": ["Deduplicate near-identical texts before splitting", "Keep all chunks/sentences from the same source document in the same split"],\n        "sheets_views": ["Ablations & Hyperparameters", "EDA Findings"],\n        "docs_focus": "Tokenizer & sequence-length sensitivity, entity/class slice performance, out-of-vocabulary / rare-class errors, and latency vs accuracy.",\n    },\n    "llm_genai": {\n        "display_name": "LLMs, Prompt Engineering & Generative AI",\n        "terminology": {"model_unit": "Foundation Model / Prompt Template / Decoding Policy", "data_unit": "Prompts / Completions / Multimodal generations", "split_strategy": "Decontaminated Eval Benchmark / Frozen Prompt Set", "tuning_knobs": "model_id, prompt_template_version, system_prompt_hash, temperature, top_p, max_new_tokens, few_shot_count, structured_schema"},\n        "tracked_parameters": ["model_id", "prompt_template_version", "few_shot_examples", "temperature", "top_p", "max_new_tokens", "judge_model_and_rubric", "response_format"],\n        "candidate_metrics": ["exact_match", "pass_at_k", "rubric_judge_score", "bleu_or_rouge_l", "bertscore", "format_validity_rate", "tokens_per_sec", "cost_per_1k_calls"],\n        "extra_context_keys": ["prompt_template_hash", "decoding_temperature", "judge_model_revision", "judge_rubric_hash"],\n        "leakage_and_split_controls": ["Never tune few-shot examples on the final test prompts", "Separate deterministic task accuracy from LLM-as-a-Judge scores and lock judge model/temperature=0", "Check format/JSON schema compliance separately from semantic quality"],\n        "sheets_views": ["Ablations & Hyperparameters", "Colab Compute & Resource Ledger"],\n        "docs_focus": "Prompt template & few-shot ablations, deterministic vs judge-based metrics, hallucination/refusal taxonomy, and latency/token cost pareto frontier.",\n    },\n    "finetuning": {\n        "display_name": "LLM / Foundation Model Fine-Tuning (SFT, PEFT, LoRA/QLoRA, DPO)",\n        "terminology": {"model_unit": "Base Checkpoint + PEFT Adapter / Merged Weights", "data_unit": "Instruction-Response pairs / Preference triples", "split_strategy": "Prompt-Family Deduplicated Train / Val / Eval", "tuning_knobs": "base_model, peft_method, lora_r, lora_alpha, lora_dropout, target_modules, quant_bits, lr, effective_batch_size, max_seq_len, chat_template, dpo_beta"},\n        "tracked_parameters": ["base_model_revision", "finetuning_method", "lora_r", "lora_alpha", "lora_dropout", "target_modules", "quantization_bits", "learning_rate", "effective_batch_size", "max_seq_length", "chat_template", "epochs_or_steps", "dpo_beta"],\n        "candidate_metrics": ["val_perplexity_or_loss", "downstream_task_score", "instruction_following_rate", "win_rate_vs_base", "catastrophic_forgetting_delta", "peak_vram_gib", "training_tokens_per_sec"],\n        "extra_context_keys": ["base_model_revision", "chat_template_hash", "quantization_bits", "eval_decoding_config"],\n        "leakage_and_split_controls": ["Decontaminate instruction tuning data against downstream evaluation benchmarks (n-gram / embedding overlap check)", "Verify exact chat template & EOS token parity between training and inference", "Track base model zero-shot baseline AND general capability retention (catastrophic forgetting check)"],\n        "sheets_views": ["Training Curves", "Ablations & Hyperparameters", "Colab Compute & Resource Ledger"],\n        "docs_focus": "Base vs LoRA/QLoRA adapter comparison, rank (r/alpha) & target-module ablations, training loss vs downstream eval curves, VRAM/Colab CU efficiency, and forgetting diagnostics.",\n    },\n    "rag": {\n        "display_name": "Retrieval-Augmented Generation (RAG, Vector/Hybrid Search, GraphRAG)",\n        "terminology": {"model_unit": "Ingestion + Retriever + Reranker + Generator Pipeline", "data_unit": "Corpus Documents -> Chunks -> Queries -> Grounded Answers", "split_strategy": "Frozen Corpus Snapshot + Disjoint Query/Document Eval Set", "tuning_knobs": "chunk_size, chunk_overlap, embedding_model, index_type, retrieval_mode, top_k_retrieve, reranker_model, top_k_rerank, generator_model, citation_prompt"},\n        "tracked_parameters": ["corpus_revision", "chunk_strategy_and_size", "chunk_overlap", "embedding_model", "hybrid_bm25_alpha", "top_k_retrieve", "reranker_model", "top_k_rerank", "generator_model", "prompt_template"],\n        "candidate_metrics": ["recall_at_k", "ndcg_at_k", "mrr_at_k", "context_precision", "faithfulness_groundedness", "answer_correctness", "citation_f1", "end_to_end_latency_ms"],\n        "extra_context_keys": ["corpus_revision", "index_revision", "top_k_cutoff", "stage_retrieval_vs_generation"],\n        "leakage_and_split_controls": ["Separate Retrieval metrics (Recall@k, MRR, nDCG) from Generation metrics (Faithfulness, Answer Correctness, Citation F1)", "Lock corpus snapshot & index revision across retriever comparisons", "Ensure evaluation queries were not synthesized directly from the top-1 chunk without multi-hop / paraphrase verification"],\n        "sheets_views": ["Ablations & Hyperparameters", "Colab Compute & Resource Ledger"],\n        "docs_focus": "Two-stage evaluation tables (Retrieval stage vs Answer/Faithfulness stage), chunking & embedding ablations, reranker impact vs latency, and unanswerable/hallucination error slices.",\n    },\n    "agentic": {\n        "display_name": "Agentic AI, Multi-Agent Workflows & Tool-Use Systems",\n        "terminology": {"model_unit": "Agent Policy / Orchestration Graph / Toolset", "data_unit": "Task Episodes -> Multi-Step Trajectories -> Tool Calls", "split_strategy": "Sandboxed Task Suite with Deterministic Environment Reset", "tuning_knobs": "planner_model, worker_models, architecture_pattern, tool_schema_version, max_steps_budget, retry_policy, memory_strategy, guardrail_policy"},\n        "tracked_parameters": ["agent_architecture", "backbone_models", "tool_schema_version", "max_steps_budget", "max_retries", "memory_or_reflection_mode", "sandbox_environment_version", "temperature"],\n        "candidate_metrics": ["episode_success_rate", "pass_at_1", "tool_call_validity_rate", "avg_steps_to_completion", "policy_violation_count", "loop_or_timeout_rate", "cost_or_tokens_per_episode", "p95_episode_latency_sec"],\n        "extra_context_keys": ["sandbox_env_revision", "tool_schema_hash", "max_steps_budget", "granularity_episode_vs_step"],\n        "leakage_and_split_controls": ["Reset sandbox/environment state deterministically before every episode", "Never conflate step-level tool call syntax success with end-to-end episode task completion", "Redact credentials and sensitive payloads from saved trajectory traces"],\n        "sheets_views": ["Agent Traces", "Ablations & Hyperparameters", "Colab Compute & Resource Ledger"],\n        "docs_focus": "End-to-end episode success vs step-level tool reliability, planner/reflection ablations, trajectory failure taxonomy (infinite loops, wrong tool args, premature stop), and token/latency cost per solved task.",\n    },\n    "multimodal": {\n        "display_name": "Multimodal AI (Vision-Language, Audio-Text, Cross-Modal)",\n        "terminology": {"model_unit": "Multimodal Encoder-Decoder / VLM", "data_unit": "Aligned Image/Audio/Video + Text tuples", "split_strategy": "Entity / Media-Asset Disjoint Split", "tuning_knobs": "vision_encoder, language_backbone, projector_type, frame_sample_rate, image_tiles, fusion_strategy"},\n        "tracked_parameters": ["vision_or_audio_encoder", "language_model", "connector_or_projector", "input_resolution_or_fps", "max_tokens", "finetuning_recipe"],\n        "candidate_metrics": ["multimodal_task_accuracy", "vqa_score", "clip_score", "cross_modal_recall_at_k", "hallucination_rate", "latency_p95_ms"],\n        "extra_context_keys": ["modality_inputs", "media_resolution_or_sampling"],\n        "leakage_and_split_controls": ["Group split by underlying media asset so multiple captions/questions on the same image/video never cross train and test", "Verify modality alignment and missing-modality fallback behavior"],\n        "sheets_views": ["Ablations & Hyperparameters", "Colab Compute & Resource Ledger"],\n        "docs_focus": "Modality contribution ablations (unimodal vs multimodal baseline), resolution/frame-sampling tradeoffs, and cross-modal hallucination analysis.",\n    },\n    "audio_speech": {\n        "display_name": "Audio & Speech Processing (ASR, TTS, Diarization, Audio Events)",\n        "terminology": {"model_unit": "Acoustic / Speech Model", "data_unit": "Waveforms / Log-Mel Spectrograms / Utterances", "split_strategy": "Speaker / Recording-Session Disjoint Split", "tuning_knobs": "sample_rate, n_mels, window_stride, backbone, language_model_decoder, spec_augment"},\n        "tracked_parameters": ["backbone", "sample_rate_hz", "feature_frontend", "spec_augment_config", "decoder_type", "chunk_duration_sec"],\n        "candidate_metrics": ["wer", "cer", "der_diarization", "pesq_or_mos", "f1_macro", "real_time_factor_rtf"],\n        "extra_context_keys": ["sample_rate_hz", "text_normalizer_version", "snr_noise_condition"],\n        "leakage_and_split_controls": ["Split strictly by speaker_id and recording session to prevent voice/channel leakage", "Apply identical text normalization before computing WER/CER"],\n        "sheets_views": ["Ablations & Hyperparameters", "Training Curves"],\n        "docs_focus": "Speaker-disjoint WER/CER results, noise/accent robustness slices, SpecAugment ablations, and Real-Time Factor (RTF) efficiency.",\n    },\n    "recommender_ranking": {\n        "display_name": "Recommender Systems & Learning-to-Rank",\n        "terminology": {"model_unit": "Candidate Retriever + Ranking Model", "data_unit": "User-Item Interactions / Query-Candidate Slates", "split_strategy": "Temporal Global Cutoff or Leave-Last-k-Out per User", "tuning_knobs": "embedding_dim, negative_sampling_ratio, loss_function, top_k_cutoff, cold_start_fallback"},\n        "tracked_parameters": ["model_architecture", "embedding_dim", "negative_sampling", "ranking_loss", "top_k_cutoff", "feature_crosses"],\n        "candidate_metrics": ["ndcg_at_k", "recall_at_k", "map_at_k", "hit_rate_at_k", "catalog_coverage", "inference_latency_ms"],\n        "extra_context_keys": ["top_k_cutoff", "negative_sampling_eval_pool", "cold_vs_warm_slice"],\n        "leakage_and_split_controls": ["Enforce global temporal cutoff so future interactions never leak into user/item history features", "Evaluate on full or consistent negative candidate pools, never biased random mini-samples without disclosure"],\n        "sheets_views": ["Ablations & Hyperparameters", "EDA Findings"],\n        "docs_focus": "Retrieval vs re-ranking stage metrics (@k), warm vs cold-start user/item slices,popularity-bias analysis, and serving latency.",\n    },\n    "custom": {\n        "display_name": "Custom / Hybrid / Scientific / Domain-Specific ML",\n        "terminology": {"model_unit": "Domain Model / Pipeline", "data_unit": "Domain observations", "split_strategy": "Domain-validated independent split", "tuning_knobs": "Task-defined experimental factors"},\n        "tracked_parameters": ["model_or_method", "experimental_factor", "key_hyperparameters", "preprocessing_version"],\n        "candidate_metrics": ["primary_objective_metric", "secondary_constraint_metric", "runtime_seconds"],\n        "extra_context_keys": ["evaluation_protocol_version"],\n        "leakage_and_split_controls": ["Establish domain unit of independence before splitting", "Separate exploratory diagnostics from ranked evaluation claims"],\n        "sheets_views": ["Ablations & Hyperparameters"],\n        "docs_focus": "Domain problem formulation, custom evaluation contract, baseline vs experimental variants, and reproducibility.",\n    },\n}\n\n\ndef resolve_domain_profile(families, *, custom_overrides=None):\n    """Synthesize a polymorphic domain profile by inheriting across 1+ ML domains.\n\n    Supports any single domain, multi-domain hybrid (e.g. [\'computer_vision\', \'finetuning\']\n    or [\'rag\', \'agentic\']), and project-specific custom_overrides for terminology,\n    tracked parameters, metrics, context keys, leakage rules, and Docs/Sheets style.\n    """\n    if not isinstance(families, list) or not families:\n        families = ["custom"]\n    cleaned = []\n    for fam in families:\n        if not isinstance(fam, str) or fam not in DOMAIN_CATALOG:\n            raise ValueError(f"Unsupported domain family: {fam!r}. Supported: {sorted(DOMAIN_CATALOG)}")\n        if fam not in cleaned:\n            cleaned.append(fam)\n\n    merged_terminology = {}\n    tracked_parameters = []\n    candidate_metrics = []\n    extra_context_keys = []\n    leakage_controls = []\n    sheets_views = []\n    docs_focus_parts = []\n    display_names = []\n\n    for fam in cleaned:\n        spec = DOMAIN_CATALOG[fam]\n        display_names.append(spec["display_name"])\n        for k, v in spec["terminology"].items():\n            merged_terminology[k] = f"{merged_terminology[k]} + {v}" if k in merged_terminology and v not in merged_terminology[k] else v\n        for item in spec["tracked_parameters"]:\n            if item not in tracked_parameters:\n                tracked_parameters.append(item)\n        for item in spec["candidate_metrics"]:\n            if item not in candidate_metrics:\n                candidate_metrics.append(item)\n        for item in spec["extra_context_keys"]:\n            if item not in extra_context_keys:\n                extra_context_keys.append(item)\n        for item in spec["leakage_and_split_controls"]:\n            if item not in leakage_controls:\n                leakage_controls.append(item)\n        for item in spec["sheets_views"]:\n            if item not in sheets_views:\n                sheets_views.append(item)\n        docs_focus_parts.append(f"[{spec[\'display_name\']}] {spec[\'docs_focus\']}")\n\n    profile = {\n        "inherited_families": cleaned,\n        "is_hybrid": len(cleaned) > 1,\n        "display_title": " × ".join(display_names),\n        "terminology": merged_terminology,\n        "tracked_parameters": tracked_parameters,\n        "candidate_metrics": candidate_metrics,\n        "extra_context_keys": extra_context_keys,\n        "leakage_and_split_controls": leakage_controls,\n        "sheets_views": sheets_views,\n        "docs_narrative_focus": " | ".join(docs_focus_parts),\n    }\n\n    if custom_overrides is not None:\n        if not isinstance(custom_overrides, dict):\n            raise ValueError("custom_overrides must be a JSON object")\n        check_json(custom_overrides)\n        if isinstance(custom_overrides.get("terminology"), dict):\n            profile["terminology"] = {**profile["terminology"], **custom_overrides["terminology"]}\n        for list_key in ("tracked_parameters", "candidate_metrics", "extra_context_keys",\n                         "leakage_and_split_controls", "sheets_views"):\n            if isinstance(custom_overrides.get(list_key), list):\n                for item in custom_overrides[list_key]:\n                    if isinstance(item, str) and item.strip() and item not in profile[list_key]:\n                        profile[list_key].append(item.strip())\n        if isinstance(custom_overrides.get("docs_narrative_focus"), str) and custom_overrides["docs_narrative_focus"].strip():\n            profile["docs_narrative_focus"] = custom_overrides["docs_narrative_focus"].strip()\n\n    check_json(profile)\n    return profile\n\n\n_DOMAIN_DETECTION_RULES = (\n    ("finetuning", {\n        "files": (r"lora", r"qlora", r"peft", r"sft", r"dpo", r"rlhf", r"adapter_config\\.json", r"instruction"),\n        "columns": ("instruction", "chosen", "rejected", "system_prompt", "conversations", "messages", "prompt_response"),\n        "imports": ("peft", "trl", "unsloth", "bitsandbytes", "axolotl"),\n        "keywords": (r"\\blora\\b", r"\\bqlora\\b", r"\\bpeft\\b", r"\\bfine[- ]?tun(?:e|ing)\\b", r"\\bsft\\b", r"\\bdpo\\b", r"\\brlhf\\b"),\n    }),\n    ("rag", {\n        "files": (r"corpus", r"chunks?", r"faiss", r"chroma", r"qdrant", r"bm25", r"retriever", r"rerank", r"knowledge_base"),\n        "columns": ("retrieved_chunks", "context_chunks", "ground_truth_answer", "chunk_id", "doc_id", "query_text"),\n        "imports": ("faiss", "chromadb", "qdrant_client", "llama_index", "ragas", "rank_bm25", "sentence_transformers"),\n        "keywords": (r"\\brag\\b", r"retrieval[- ]augmented", r"\\brecall@\\d+", r"\\bndcg@\\d+", r"\\bfaithfulness\\b", r"\\brerank(?:er|ing)\\b", r"\\bchunk(?:ing|_size)\\b"),\n    }),\n    ("agentic", {\n        "files": (r"agent", r"trajector(?:y|ies)", r"episodes?", r"tool_schema", r"sandbox", r"mcp"),\n        "columns": ("episode_id", "trajectory", "tool_calls", "observation", "step_budget", "agent_action"),\n        "imports": ("langgraph", "autogen", "crewai", "smolagents", "gymnasium"),\n        "keywords": (r"\\bagentic\\b", r"\\bmulti[- ]agent\\b", r"\\btool[- ]call(?:ing|s)?\\b", r"\\bepisode_success\\b", r"\\btrajectory\\b", r"\\breact\\s+(?:agent|loop|prompt)\\b"),\n    }),\n    ("computer_vision", {\n        "files": (r"\\.(?:jpg|jpeg|png|bmp|tif|tiff|webp|mp4|avi)\\Z", r"coco", r"yolo", r"bboxes", r"segmentation", r"masks?", r"images\\.zip"),\n        "columns": ("image_path", "image_id", "bbox", "xmin", "ymin", "xmax", "ymax", "rle_mask", "mask_path", "frame_id"),\n        "imports": ("cv2", "timm", "ultralytics", "torchvision", "albumentations", "kornia", "mmdet"),\n        "keywords": (r"\\bcomputer vision\\b", r"\\bobject detection\\b", r"\\bsegmentation\\b", r"\\bmap@\\d+|\\bmean average precision\\b", r"\\biou\\b", r"\\bdice score\\b", r"\\byolo\\b"),\n    }),\n    ("time_series", {\n        "files": (r"timeseries", r"forecast", r"horizon", r"ohlcv", r"seasonal", r"telemetry"),\n        "columns": ("timestamp", "datetime", "date", "ds", "forecast_horizon", "series_id", "lag_1", "lookback"),\n        "imports": ("statsforecast", "neuralforecast", "prophet", "pmdarima", "tsfresh", "sktime", "darts"),\n        "keywords": (r"\\btime[- ]series\\b", r"\\bforecast(?:ing)?\\b", r"\\bmase\\b", r"\\bsmape\\b", r"\\bwalk[- ]forward\\b", r"\\bhorizon\\b", r"\\bseasonality\\b"),\n    }),\n    ("audio_speech", {\n        "files": (r"\\.(?:wav|mp3|flac|ogg|m4a)\\Z", r"diarization", r"utterances?", r"spectrogram"),\n        "columns": ("audio_path", "speaker_id", "sample_rate", "duration_sec", "utterance_id", "waveform"),\n        "imports": ("librosa", "torchaudio", "soundfile", "whisper", "pyannote", "speechbrain"),\n        "keywords": (r"\\basr\\b", r"\\bspeech[- ]to[- ]text\\b", r"\\bdiarization\\b", r"\\bwer\\b", r"\\bcer\\b", r"\\blog[- ]mel\\b", r"\\bspecaugment\\b"),\n    }),\n    ("multimodal", {\n        "files": (r"multimodal", r"vqa", r"vlm", r"image_text", r"video_caption", r"cross_modal"),\n        "columns": ("image_caption", "visual_question", "audio_transcript", "modality"),\n        "imports": ("open_clip", "llava", "qwen_vl_utils"),\n        "keywords": (r"\\bmultimodal\\b", r"\\bvlm\\b", r"\\bvision[- ]language\\b", r"\\bvqa\\b", r"\\bclip[- ]?score\\b", r"\\bcross[- ]modal\\b"),\n    }),\n    ("recommender_ranking", {\n        "files": (r"interactions", r"user_item", r"clickstream", r"impressions", r"movielens", r"recsys"),\n        "columns": ("user_id", "item_id", "rating", "impression_id", "candidate_id", "slate_id"),\n        "imports": ("lightfm", "implicit", "recbole", "surprise"),\n        "keywords": (r"\\brecommender\\b", r"\\brecsys\\b", r"\\blearning[- ]to[- ]rank\\b", r"\\bcollaborative filtering\\b", r"\\bcold[- ]start\\b", r"\\bhit_rate@\\d+"),\n    }),\n    ("llm_genai", {\n        "files": (r"prompts?", r"few_shot", r"rubric", r"completions", r"generations"),\n        "columns": ("prompt", "completion", "response", "rubric_score", "judge_rationale"),\n        "imports": ("openai", "anthropic", "google.genai", "vllm", "litellm"),\n        "keywords": (r"\\bprompt engineering\\b", r"\\bfew[- ]shot\\b", r"\\bllm[- ]as[- ]a[- ]judge\\b", r"\\bpass@\\d+", r"\\bgenerative ai\\b"),\n    }),\n    ("nlp", {\n        "files": (r"conll", r"ner", r"sentiment", r"token_classification", r"text_classification"),\n        "columns": ("text", "sentence", "tokens", "ner_tags", "span_start", "span_end", "document_text"),\n        "imports": ("spacy", "nltk", "seqeval"),\n        "keywords": (r"\\bner\\b", r"\\bnamed entity\\b", r"\\btext classification\\b", r"\\bsentiment\\b", r"\\bspan[- ]f1\\b", r"\\btokenizer\\b"),\n    }),\n    ("deep_learning", {\n        "files": (r"\\.(?:pt|pth|ckpt|safetensors|onnx|h5)\\Z", r"backbone", r"checkpoints?"),\n        "columns": (),\n        "imports": ("torch", "pytorch_lightning", "lightning", "tensorflow", "keras", "jax", "flax"),\n        "keywords": (r"\\bdeep learning\\b", r"\\bneural network\\b", r"\\bbackprop\\b", r"\\badamw\\b", r"\\bcosine annealing\\b"),\n    }),\n    ("unsupervised", {\n        "files": (r"clustering", r"anomaly", r"outliers?", r"unsupervised", r"embeddings"),\n        "columns": ("cluster_id", "anomaly_score", "is_outlier"),\n        "imports": ("umap", "hdbscan"),\n        "keywords": (r"\\bunsupervised\\b", r"\\bclustering\\b", r"\\banomaly detection\\b", r"\\bsilhouette\\b", r"\\bdimensionality reduction\\b"),\n    }),\n    ("classical", {\n        "files": (r"\\.(?:csv|tsv|parquet|feather)\\Z", r"train\\.csv", r"test\\.csv", r"sample_submission\\.csv"),\n        "columns": ("target", "label", "class", "y", "prediction"),\n        "imports": ("sklearn", "scikit-learn", "xgboost", "lightgbm", "catboost", "optuna"),\n        "keywords": (r"\\btabular\\b", r"\\bxgboost\\b", r"\\blightgbm\\b", r"\\bcatboost\\b", r"\\bgradient boosting\\b", r"\\brandom forest\\b", r"\\broc[- ]auc\\b", r"\\bpr[- ]auc\\b"),\n    }),\n)\n\n\ndef auto_detect_domain(*, inventory=None, csv_headers=None, rules_text="", code_imports=None, custom_overrides=None):\n    """Auto-detect project ML domain(s), data schema signals, and inherited domain profile.\n\n    Inspects Drive file inventory names, CSV/table headers, rules/guidelines text,\n    and code imports without executing any untrusted files.\n    """\n    inventory = inventory or []\n    csv_headers = csv_headers or []\n    code_imports = code_imports or []\n    rules_text = rules_text or ""\n    if (not isinstance(inventory, list) or not isinstance(csv_headers, list)\n            or not isinstance(code_imports, list) or not isinstance(rules_text, str)):\n        raise ValueError("auto_detect_domain expects list inventory/csv_headers/code_imports and string rules_text")\n\n    file_names = []\n    for item in inventory:\n        if isinstance(item, dict) and isinstance(item.get("name"), str):\n            file_names.append(item["name"].strip().lower())\n        elif isinstance(item, str):\n            file_names.append(item.strip().lower())\n\n    norm_headers = [h.strip().lower() for h in csv_headers if isinstance(h, str) and h.strip()]\n    norm_imports = [imp.strip().lower() for imp in code_imports if isinstance(imp, str) and imp.strip()]\n    lower_rules = rules_text.lower()\n\n    evidence_by_family = {}\n    scores = {}\n\n    for family, matchers in _DOMAIN_DETECTION_RULES:\n        reasons = []\n        weight = 0\n        for pat in matchers["files"]:\n            matched_file = next((fn for fn in file_names if re.search(pat, fn, re.I)), None)\n            if matched_file:\n                reasons.append(f"file:{matched_file}")\n                weight += 2 if family != "classical" else 1\n        for col in matchers["columns"]:\n            if col in norm_headers:\n                reasons.append(f"column:{col}")\n                weight += 3\n        for imp in matchers["imports"]:\n            if any(imp == x or x.startswith(imp + ".") for x in norm_imports):\n                reasons.append(f"import:{imp}")\n                weight += 3\n        for kw in matchers["keywords"]:\n            if re.search(kw, lower_rules, re.I):\n                clean_kw = kw.replace(r"\\b", "").replace("\\\\", "")\n                reasons.append(f"rule:{clean_kw}")\n                weight += 3\n        if reasons:\n            evidence_by_family[family] = reasons\n            scores[family] = weight\n\n    # Schema signal extraction for leakage & split controls\n    temporal_cols = [h for h in norm_headers if re.search(r"(?:^|_)(?:timestamp|datetime|date|time|ds|year|month|week|day)(?:$|_)", h)]\n    group_cols = [h for h in norm_headers if re.search(r"(?:^|_)(?:user_id|patient_id|speaker_id|entity_id|group_id|session_id|customer_id|account_id|scene_id|video_id|doc_id)(?:$|_)", h)]\n    media_cols = [h for h in norm_headers if re.search(r"(?:image|audio|video|frame|mask|bbox|waveform)_(?:path|url|id)", h)]\n    text_cols = [h for h in norm_headers if re.search(r"(?:^|_)(?:text|prompt|instruction|query|question|sentence|document|transcript|completion|response)(?:$|_)", h)]\n    target_cols = [h for h in norm_headers if re.search(r"(?:^|_)(?:target|label|class|y|is_fraud|outcome|score|churn)(?:$|_)", h)]\n\n    # If user_id + item_id coexist in headers, strongly boost recommender_ranking\n    if "user_id" in norm_headers and any(c in norm_headers for c in ("item_id", "product_id", "movie_id", "candidate_id")):\n        evidence_by_family.setdefault("recommender_ranking", []).append("schema:user_item_pair")\n        scores["recommender_ranking"] = scores.get("recommender_ranking", 0) + 5\n\n    # If temporal columns + forecasting keywords or horizon exist, boost time_series\n    if temporal_cols and ("time_series" in scores or any("lag" in h or "horizon" in h for h in norm_headers)):\n        evidence_by_family.setdefault("time_series", []).append(f"schema:temporal({temporal_cols[0]})")\n        scores["time_series"] = scores.get("time_series", 0) + 3\n\n    ranked = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))\n    if not ranked:\n        detected = ["custom"]\n        confidence = "none"\n    else:\n        top_score = ranked[0][1]\n        # Keep specialized families within strong evidence threshold; don\'t let generic CSV alone override a specific domain\n        selected = [fam for fam, sc in ranked if sc >= max(3, top_score - 3)]\n        if not selected:\n            selected = [ranked[0][0]]\n        if len(selected) > 1 and "classical" in selected and any(\n            f in selected for f in ("computer_vision", "time_series", "finetuning", "rag", "agentic", "audio_speech", "multimodal", "recommender_ranking")\n        ):\n            if scores.get("classical", 0) <= 2:\n                selected.remove("classical")\n        detected = selected[:3]\n        confidence = "high" if top_score >= 5 else "medium" if top_score >= 3 else "low"\n\n    profile = resolve_domain_profile(detected, custom_overrides=custom_overrides)\n    return {\n        "detected_families": detected,\n        "confidence": confidence,\n        "is_hybrid": len(detected) > 1,\n        "evidence_by_family": {k: evidence_by_family[k] for k in detected if k in evidence_by_family},\n        "all_candidate_scores": dict(ranked),\n        "schema_signals": {\n            "temporal_columns": temporal_cols,\n            "entity_or_group_columns": group_cols,\n            "media_columns": media_cols,\n            "text_or_prompt_columns": text_cols,\n            "candidate_target_columns": target_cols,\n        },\n        "domain_profile": profile,\n    }\n\n\n\ndef default_project_memory(project_id):\n    """Create an initial combined project-specific memory record (never global)."""\n    project_id = component(project_id)\n    return {\n        "schema_version": PROJECT_MEMORY_SCHEMA,\n        "memory_scope": "project_specific_only",\n        "global_memory_prohibited": True,\n        "project_id": project_id,\n        "updated_at_utc": utc_now(),\n        "last_writer_skill": "ml-project-setup",\n        "setup_context": {\n            "root_folder_id": None,\n            "task_families": [],\n            "domain_profile": None,\n            "objective": None,\n            "primary_metric_id": None,\n            "split_and_leakage_policy": None,\n            "folder_slots": {},\n            "hf_selected_assets": [],\n            "library_docs_verifications": [],\n        },\n        "destinations": {\n            "tracker_sheet_id": None,\n            "report_doc_id": None,\n            "presentation_slides_id": None,\n            "active_notebook_id": None,\n        },\n        "active_lineage": {\n            "baseline_run_id": None,\n            "baseline_revision_id": None,\n            "best_run_id": None,\n            "best_revision_id": None,\n            "best_metric_value": None,\n            "completed_run_count": 0,\n        },\n        "experiment_memory": {\n            "hyperparameter_lessons": [],\n            "ablation_findings": [],\n            "failed_runs_and_pitfalls": [],\n            "compute_units_total": 0.0,\n        },\n        "publication_memory": {\n            "sheets_last_operation_id": None,\n            "sheets_preserved_regions": [],\n            "docs_last_operation_id": None,\n            "docs_preserved_sections": [],\n            "slides_last_operation_id": None,\n            "slides_preserved_slide_ids": [],\n            "submission_validation_status": "not_run",\n            "validated_submission_artifacts": [],\n            "last_quality_gate_status": "not_run",\n        },\n        "next_experiments_queue": [],\n    }\n\n\ndef validate_project_memory(memory, *, project_id=None):\n    """Validate project-specific memory isolation, schema, and JSON safety."""\n    check_json(memory)\n    if not isinstance(memory, dict) or memory.get("schema_version") != PROJECT_MEMORY_SCHEMA:\n        raise ValueError("Unsupported project memory schema")\n    if memory.get("memory_scope") != "project_specific_only" or memory.get("global_memory_prohibited") is not True:\n        raise ValueError("Project memory must remain strictly project-specific (never global)")\n    pid = component(memory.get("project_id"))\n    if project_id is not None and pid != component(project_id):\n        raise ValueError("Cross-project memory contamination rejected")\n    for section in ("setup_context", "destinations", "active_lineage", "experiment_memory", "publication_memory"):\n        if not isinstance(memory.get(section), dict):\n            raise ValueError(f"Project memory section {section} must be an object")\n    if not isinstance(memory.get("next_experiments_queue"), list):\n        raise ValueError("next_experiments_queue must be a list")\n    return memory\n\n\ndef load_project_memory(project_root, *, project_id=None):\n    """Load combined project-specific memory from 05_reports/autopilot/project_memory.json."""\n    path = within(project_root, PROJECT_MEMORY_RELPATH)\n    if not path.is_file():\n        if project_id is None:\n            raise FileNotFoundError(str(path))\n        return default_project_memory(project_id)\n    return validate_project_memory(read_json(path), project_id=project_id)\n\n\ndef update_project_memory(project_root, project_id, *, skill_name, patch):\n    """Atomically update combined project-specific memory inside the project root."""\n    project_id = component(project_id)\n    skill_name = component(skill_name)\n    if not isinstance(patch, dict) or not patch or any(k not in _MEMORY_SECTIONS for k in patch):\n        raise ValueError("Memory patch must target valid project memory sections")\n    check_json(patch)\n    path = within(project_root, PROJECT_MEMORY_RELPATH)\n    expected = file_digest(path) if path.is_file() else None\n    current = load_project_memory(project_root, project_id=project_id)\n    for section, updates in patch.items():\n        if section == "next_experiments_queue":\n            if not isinstance(updates, list):\n                raise ValueError("next_experiments_queue patch must be a list")\n            current[section] = list(updates)\n        else:\n            if not isinstance(updates, dict):\n                raise ValueError(f"Section {section} patch must be an object")\n            current[section] = {**current[section], **updates}\n    current["updated_at_utc"] = utc_now()\n    current["last_writer_skill"] = skill_name\n    validate_project_memory(current, project_id=project_id)\n    atomic_json(path, current, expected=expected)\n    return current\n\n\n\n'
helper = Path('/content') / ('autopilot_runtime_' + CONFIG['project_id'] + '.py')
helper.write_text(RUNTIME_SOURCE, encoding='utf-8')
spec = importlib.util.spec_from_file_location('autopilot_runtime', helper)
autopilot = importlib.util.module_from_spec(spec)
spec.loader.exec_module(autopilot)
autopilot.component(RUN_ID)
missing = []
for pin in ([] if RECOVER_RUN_ID else CONFIG.get('packages', [])):
    name, version = pin.split('==', 1)
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(pin)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade-strategy', 'only-if-needed', *missing], check=True)
    raise RuntimeError('Dependencies installed. Restart the Colab runtime, then run from the top.')
DEPENDENCIES = ({'status': 'recovery_only'} if RECOVER_RUN_ID else autopilot.dependency_report(CONFIG.get('packages', []), policy=CONFIG.get('dependency_check', 'required')))
if DEPENDENCIES['status'] == 'unrelated_conflicts':
    print('Environment note:', DEPENDENCIES['unrelated_conflicts'], 'preinstalled conflicts outside this workload; no packages changed.')


Run ID: 001798f3-a360-479d-a2bc-7d389ccf3203
Environment note: 1 preinstalled conflicts outside this workload; no packages changed.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = autopilot.resolve_project_root('/content/drive', CONFIG['drive_relative_path'], CONFIG['project_id'], override=PROJECT_PATH_OVERRIDE, folder_id=CONFIG.get('drive_folder_id'))
print('Project:', PROJECT_ROOT)


Mounted at /content/drive
Project: /content/drive/MyDrive/DataSprint_2.0_Road_Damage


In [3]:
import shutil, platform, hashlib, importlib.util
SPOOL_ROOT = Path('/content/autopilot_spool') / CONFIG['project_id']
if RECOVER_RUN_ID:
    recovered = autopilot.recover_spooled_events(PROJECT_ROOT, SPOOL_ROOT, RUN_ID, project_id=CONFIG['project_id'])
    print('Recovered', len(recovered), 'staged events. No experiment was rerun.')
    raise RuntimeError('Recovery-only mode finished. Request a project sync; do not run the task cell.')
free_gib = shutil.disk_usage('/content').free / 2**30
if free_gib < CONFIG.get('min_free_disk_gib', 0):
    raise RuntimeError('Insufficient local disk for this task')
gpu_query = shutil.which('nvidia-smi')
gpu_description = None
if gpu_query and (CONFIG.get('gpu_required') or CONFIG.get('min_gpu_memory_gib', 0) > 0):
    gpu_description = subprocess.check_output([gpu_query, '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], text=True).strip()
if CONFIG.get('gpu_required') or CONFIG.get('min_gpu_memory_gib', 0) > 0:
    if not gpu_description:
        raise RuntimeError('Select a compatible GPU runtime in Colab')
    memories = [float(line.rsplit(',', 1)[1].strip()) / 1024 for line in gpu_description.splitlines()]
    if max(memories) < CONFIG.get('min_gpu_memory_gib', 0):
        raise RuntimeError('GPU memory below configured minimum')
ENVIRONMENT = {'python': platform.python_version(), 'platform': platform.platform(), 'gpu': gpu_description, 'packages': {pin.split('==')[0]: importlib.metadata.version(pin.split('==')[0]) for pin in CONFIG.get('packages', [])}}
CHECKPOINT_CONFIG = {'project_id': CONFIG['project_id'], 'experiment': CONFIG.get('experiment', {}), 'evaluation': CONFIG.get('context', {}), 'metric_context': CONFIG.get('metric_context', {}), 'environment': ENVIRONMENT, 'source_fingerprints': CONFIG.get('source_fingerprints', {})}
if RESUME_RUN_ID:
    manifest = CONFIG.get('resume_manifest')
    if not manifest:
        raise RuntimeError('Set resume_manifest and restore task-specific optimizer/RNG/model state explicitly')
    autopilot.verify_checkpoint(PROJECT_ROOT, autopilot.within(PROJECT_ROOT, manifest), CHECKPOINT_CONFIG, run_id=RUN_ID)
print('Preflight passed. Local free GiB:', round(free_gib, 1))


Preflight passed. Local free GiB: 235.6


## Task and logging
The task must assign `metrics` from actual observations and call `check_budget()` at safe loop boundaries. Write model, optimizer, scheduler, step, and RNG state to finalized checkpoint artifacts, then call `autopilot.save_checkpoint_manifest(PROJECT_ROOT, RUN_ID, checkpoint_id, CHECKPOINT_CONFIG, files)`. Restore that state explicitly on resume. The helper verifies artifacts; it does not serialize models. Budget checks cannot interrupt a blocking GPU operation. Estimated cost is user-supplied and does not cap Colab billing. After a completed run, start a fresh run ID for another experiment.


In [4]:
if globals().get('RECOVER_RUN_ID'):
    raise RuntimeError('Recovery-only mode cannot execute the experiment task.')
EVENT_DIRS = [root / '03_results' / 'runs' / RUN_ID / 'events' for root in (PROJECT_ROOT, SPOOL_ROOT)]
if any((directory / 'final.json').exists() for directory in EVENT_DIRS):
    raise RuntimeError('This run already completed or has a staged final result. Use RECOVER_RUN_ID for delivery, or START_NEW_RUN for an independent attempt.')
if not RESUME_RUN_ID and any(directory.exists() and any(directory.glob('start-*.json')) for directory in EVENT_DIRS):
    raise RuntimeError('An attempt already exists for this run. Use a fresh RUN_ID or explicitly resume a verified checkpoint.')
STARTED = time.monotonic()
metrics = {}
TASK_ARTIFACTS = {}
def check_budget():
    limit = CONFIG.get('budget', {}).get('max_hours')
    if limit is not None and time.monotonic() - STARTED > limit * 3600:
        raise RuntimeError('Runtime budget reached; save a checkpoint and stop')
autopilot.log_experiment(PROJECT_ROOT, CONFIG.get('experiment_name', 'experiment'), {}, run_id=RUN_ID, event_id='start-' + str(uuid.uuid4()), status='partial', spool_root=SPOOL_ROOT, config={'project_id': CONFIG['project_id'], 'experiment': CONFIG.get('experiment', {}), 'environment': ENVIRONMENT, 'source_fingerprints': CONFIG.get('source_fingerprints', {})})
_STAGE_STATE = {'next': 0, 'failed': False, 'finalized': False}
def _execute_stage(index, source):
    if _STAGE_STATE['failed'] or _STAGE_STATE['finalized'] or index != _STAGE_STATE['next']:
        raise RuntimeError('Run task sections once in order; recover or create a new run after failure')
    try:
        check_budget()
        exec(compile(source, '<experiment-stage>', 'exec'), globals())
        check_budget()
    except autopilot.LogDeliveryError:
        _STAGE_STATE['failed'] = True
        print('Delivery pending; preserve spool and recover without repeating compute:', RUN_ID)
        raise
    except BaseException as error:
        _STAGE_STATE['failed'] = True
        status = 'interrupted' if isinstance(error, KeyboardInterrupt) else 'failed'
        try:
            autopilot.log_experiment(PROJECT_ROOT, CONFIG.get('experiment_name', 'experiment'), {}, run_id=RUN_ID, event_id='failure-' + str(uuid.uuid4()), status=status, spool_root=SPOOL_ROOT, config={'project_id': CONFIG['project_id'], 'experiment': CONFIG.get('experiment', {}), 'environment': ENVIRONMENT, 'source_fingerprints': CONFIG.get('source_fingerprints', {})}, errors=[{'type': type(error).__name__, 'stage': index}])
        except Exception:
            print('Failure delivery pending; preserve local runtime files.')
        raise
    _STAGE_STATE['next'] += 1


## 1. A100/L4 TF32 Preflight, Resource Auto-Scaling & API Dry-Run
Enable Ampere/Ada TF32 Tensor Cores & cuDNN autotuner, auto-scale batch/workers/cache from VRAM/RAM/Disk, validate Ultralytics & nested Albumentations OneOf bbox pipeline, and verify all 21,041 images on Drive.

Read-only source preview. Edit the task/section input and rebuild; changing this Markdown does not change execution. The code cell below executes this source verbatim:
```python
import inspect
import json
import os
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import albumentations as A
import numpy as np
import pandas as pd
import torch
from PIL import Image
from ultralytics import YOLO
from ultralytics.cfg import DEFAULT_CFG_DICT

EXP_CFG = CONFIG['experiment']
SEED = int(EXP_CFG.get('seed', 42))
SEED_RECEIPT = autopilot.seed_everything(SEED, deterministic=False)
CONFIG['experiment']['seed_receipt'] = SEED_RECEIPT

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required for v3 training. Select A100, L4, or T4 in Colab Runtime.')

# Enable Ampere/Ada TF32 Tensor Cores and cuDNN kernel autotuning
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True

def _avail_ram_gib() -> float:
    try:
        return (os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_AVPHYS_PAGES')) / (1024 ** 3)
    except Exception:
        for line in Path('/proc/meminfo').read_text(encoding='utf-8').splitlines():
            if line.startswith('MemAvailable:'):
                return float(line.split()[1]) / (1024 ** 2)
    return 12.0

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
FREE_RAM_GIB = _avail_ram_gib()
FREE_DISK_GIB = shutil.disk_usage('/content').free / (1024 ** 3)
CPU_CORES = os.cpu_count() or 4

if bool(EXP_CFG.get('auto_batch_by_vram', True)):
    if GPU_VRAM_GIB >= 36.0:
        EFFECTIVE_BATCH = int(EXP_CFG.get('batch_a100', 64))
        DATALOADER_WORKERS = min(12, CPU_CORES)
        GPU_TIER = 'A100'
    elif GPU_VRAM_GIB >= 20.0:
        EFFECTIVE_BATCH = int(EXP_CFG.get('batch_l4', 32))
        DATALOADER_WORKERS = min(8, CPU_CORES)
        GPU_TIER = 'L4'
    else:
        EFFECTIVE_BATCH = int(EXP_CFG.get('batch_t4', 16))
        DATALOADER_WORKERS = min(4, CPU_CORES)
        GPU_TIER = 'T4'
else:
    EFFECTIVE_BATCH = int(EXP_CFG.get('batch_l4', 32))
    DATALOADER_WORKERS = min(8, CPU_CORES)
    GPU_TIER = 'CUSTOM'

# Safe cache selection: RAM cache only on High-RAM (>=60 GiB free); SSD disk cache when >=45 GiB free
CACHE_MODE = 'ram' if FREE_RAM_GIB >= 60.0 else ('disk' if FREE_DISK_GIB >= 45.0 else False)
print(f'PyTorch: {torch.__version__} | GPU: {GPU_NAME} ({GPU_VRAM_GIB:.1f} GiB -> {GPU_TIER})')
print(f'RAM free: {FREE_RAM_GIB:.1f} GiB | Disk free: {FREE_DISK_GIB:.1f} GiB | CPUs: {CPU_CORES}')
print(f'Auto config -> batch={EFFECTIVE_BATCH}, workers={DATALOADER_WORKERS}, cache={CACHE_MODE!r}, TF32=ON')

# Fast preflight API & bbox-pipeline dry-run (<0.2s)
assert any(p.kind == inspect.Parameter.VAR_KEYWORD for p in inspect.signature(YOLO.train).parameters.values())
assert any(p.kind == inspect.Parameter.VAR_KEYWORD for p in inspect.signature(YOLO.predict).parameters.values())
assert all(k in DEFAULT_CFG_DICT for k in ('conf', 'iou', 'max_det', 'cos_lr', 'close_mosaic', 'fliplr', 'flipud', 'degrees', 'amp', 'cache', 'agnostic_nms'))
FP16_KEY = 'quantize' if 'quantize' in DEFAULT_CFG_DICT else 'half'
FP16_VAL = 16 if FP16_KEY == 'quantize' else True

_dummy_img = np.zeros((64, 64, 3), dtype=np.uint8)
_aug_test = [
    A.OneOf([
        A.ImageCompression(compression_type='jpeg', quality_range=(6, 22), p=1.0),
        A.Compose([
            A.RandomBrightnessContrast(brightness_limit=(-0.75, -0.45), contrast_limit=(-0.35, 0.0), p=1.0),
            A.RandomGamma(gamma_limit=(140, 220), p=0.5),
        ], p=1.0),
        A.OneOf([
            A.GaussianBlur(blur_limit=(7, 15), p=1.0),
            A.MotionBlur(blur_limit=(7, 15), p=1.0),
        ], p=1.0),
        A.GaussNoise(std_range=(0.05, 0.15), p=1.0),
    ], p=float(EXP_CFG.get('robustness_aug_p', 0.30))),
]
_dry_comp = A.Compose(_aug_test, bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))
_dry_res = _dry_comp(image=_dummy_img, bboxes=[[10.0, 10.0, 40.0, 40.0]], class_labels=[0])
assert _dry_res['image'].shape == _dummy_img.shape and len(_dry_res['bboxes']) == 1
print(f'API & Albumentations 70/30 OneOf bbox dry-run passed (FP16 arg: {FP16_KEY}={FP16_VAL}).')

DRIVE_DATA_DIR = PROJECT_ROOT / '01_data'
RUN_DRIVE_DIR = PROJECT_ROOT / '03_results' / 'runs' / RUN_ID
RUN_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

required_csvs = ['train_labels.csv', 'val_labels.csv', 'val_image_ids.csv', 'test_image_ids.csv']
for csv_name in required_csvs:
    if not (DRIVE_DATA_DIR / csv_name).is_file():
        raise FileNotFoundError(f'Missing {csv_name} in {DRIVE_DATA_DIR}')

def _count_jpgs(folder: Path) -> int:
    return sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg'}) if folder.is_dir() else 0

train_count = _count_jpgs(DRIVE_DATA_DIR / 'train_images')
val_count = _count_jpgs(DRIVE_DATA_DIR / 'val_images')
test_count = _count_jpgs(DRIVE_DATA_DIR / 'test_images')
print(f'Drive image counts -> train: {train_count}, val: {val_count}, test: {test_count}')
if train_count < 14691 or val_count < 2150 or test_count < 4200:
    raise RuntimeError(f'Expected full dataset (14691/2150/4200), got train={train_count}, val={val_count}, test={test_count}')
```


In [5]:
_execute_stage(0, "import inspect\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport albumentations as A\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom ultralytics import YOLO\nfrom ultralytics.cfg import DEFAULT_CFG_DICT\n\nEXP_CFG = CONFIG['experiment']\nSEED = int(EXP_CFG.get('seed', 42))\nSEED_RECEIPT = autopilot.seed_everything(SEED, deterministic=False)\nCONFIG['experiment']['seed_receipt'] = SEED_RECEIPT\n\nif not torch.cuda.is_available():\n    raise RuntimeError('CUDA GPU is required for v3 training. Select A100, L4, or T4 in Colab Runtime.')\n\n# Enable Ampere/Ada TF32 Tensor Cores and cuDNN kernel autotuning\ntorch.backends.cuda.matmul.allow_tf32 = True\ntorch.backends.cudnn.allow_tf32 = True\ntorch.set_float32_matmul_precision('high')\ntorch.backends.cudnn.benchmark = True\n\ndef _avail_ram_gib() -> float:\n    try:\n        return (os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_AVPHYS_PAGES')) / (1024 ** 3)\n    except Exception:\n        for line in Path('/proc/meminfo').read_text(encoding='utf-8').splitlines():\n            if line.startswith('MemAvailable:'):\n                return float(line.split()[1]) / (1024 ** 2)\n    return 12.0\n\nGPU_NAME = torch.cuda.get_device_name(0)\nGPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)\nFREE_RAM_GIB = _avail_ram_gib()\nFREE_DISK_GIB = shutil.disk_usage('/content').free / (1024 ** 3)\nCPU_CORES = os.cpu_count() or 4\n\nif bool(EXP_CFG.get('auto_batch_by_vram', True)):\n    if GPU_VRAM_GIB >= 36.0:\n        EFFECTIVE_BATCH = int(EXP_CFG.get('batch_a100', 64))\n        DATALOADER_WORKERS = min(12, CPU_CORES)\n        GPU_TIER = 'A100'\n    elif GPU_VRAM_GIB >= 20.0:\n        EFFECTIVE_BATCH = int(EXP_CFG.get('batch_l4', 32))\n        DATALOADER_WORKERS = min(8, CPU_CORES)\n        GPU_TIER = 'L4'\n    else:\n        EFFECTIVE_BATCH = int(EXP_CFG.get('batch_t4', 16))\n        DATALOADER_WORKERS = min(4, CPU_CORES)\n        GPU_TIER = 'T4'\nelse:\n    EFFECTIVE_BATCH = int(EXP_CFG.get('batch_l4', 32))\n    DATALOADER_WORKERS = min(8, CPU_CORES)\n    GPU_TIER = 'CUSTOM'\n\n# Safe cache selection: RAM cache only on High-RAM (>=60 GiB free); SSD disk cache when >=45 GiB free\nCACHE_MODE = 'ram' if FREE_RAM_GIB >= 60.0 else ('disk' if FREE_DISK_GIB >= 45.0 else False)\nprint(f'PyTorch: {torch.__version__} | GPU: {GPU_NAME} ({GPU_VRAM_GIB:.1f} GiB -> {GPU_TIER})')\nprint(f'RAM free: {FREE_RAM_GIB:.1f} GiB | Disk free: {FREE_DISK_GIB:.1f} GiB | CPUs: {CPU_CORES}')\nprint(f'Auto config -> batch={EFFECTIVE_BATCH}, workers={DATALOADER_WORKERS}, cache={CACHE_MODE!r}, TF32=ON')\n\n# Fast preflight API & bbox-pipeline dry-run (<0.2s)\nassert any(p.kind == inspect.Parameter.VAR_KEYWORD for p in inspect.signature(YOLO.train).parameters.values())\nassert any(p.kind == inspect.Parameter.VAR_KEYWORD for p in inspect.signature(YOLO.predict).parameters.values())\nassert all(k in DEFAULT_CFG_DICT for k in ('conf', 'iou', 'max_det', 'cos_lr', 'close_mosaic', 'fliplr', 'flipud', 'degrees', 'amp', 'cache', 'agnostic_nms'))\nFP16_KEY = 'quantize' if 'quantize' in DEFAULT_CFG_DICT else 'half'\nFP16_VAL = 16 if FP16_KEY == 'quantize' else True\n\n_dummy_img = np.zeros((64, 64, 3), dtype=np.uint8)\n_aug_test = [\n    A.OneOf([\n        A.ImageCompression(compression_type='jpeg', quality_range=(6, 22), p=1.0),\n        A.Compose([\n            A.RandomBrightnessContrast(brightness_limit=(-0.75, -0.45), contrast_limit=(-0.35, 0.0), p=1.0),\n            A.RandomGamma(gamma_limit=(140, 220), p=0.5),\n        ], p=1.0),\n        A.OneOf([\n            A.GaussianBlur(blur_limit=(7, 15), p=1.0),\n            A.MotionBlur(blur_limit=(7, 15), p=1.0),\n        ], p=1.0),\n        A.GaussNoise(std_range=(0.05, 0.15), p=1.0),\n    ], p=float(EXP_CFG.get('robustness_aug_p', 0.30))),\n]\n_dry_comp = A.Compose(_aug_test, bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))\n_dry_res = _dry_comp(image=_dummy_img, bboxes=[[10.0, 10.0, 40.0, 40.0]], class_labels=[0])\nassert _dry_res['image'].shape == _dummy_img.shape and len(_dry_res['bboxes']) == 1\nprint(f'API & Albumentations 70/30 OneOf bbox dry-run passed (FP16 arg: {FP16_KEY}={FP16_VAL}).')\n\nDRIVE_DATA_DIR = PROJECT_ROOT / '01_data'\nRUN_DRIVE_DIR = PROJECT_ROOT / '03_results' / 'runs' / RUN_ID\nRUN_DRIVE_DIR.mkdir(parents=True, exist_ok=True)\n\nrequired_csvs = ['train_labels.csv', 'val_labels.csv', 'val_image_ids.csv', 'test_image_ids.csv']\nfor csv_name in required_csvs:\n    if not (DRIVE_DATA_DIR / csv_name).is_file():\n        raise FileNotFoundError(f'Missing {csv_name} in {DRIVE_DATA_DIR}')\n\ndef _count_jpgs(folder: Path) -> int:\n    return sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg'}) if folder.is_dir() else 0\n\ntrain_count = _count_jpgs(DRIVE_DATA_DIR / 'train_images')\nval_count = _count_jpgs(DRIVE_DATA_DIR / 'val_images')\ntest_count = _count_jpgs(DRIVE_DATA_DIR / 'test_images')\nprint(f'Drive image counts -> train: {train_count}, val: {val_count}, test: {test_count}')\nif train_count < 14691 or val_count < 2150 or test_count < 4200:\n    raise RuntimeError(f'Expected full dataset (14691/2150/4200), got train={train_count}, val={val_count}, test={test_count}')")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
PyTorch: 2.14.1+cu130 | GPU: NVIDIA L4 (22.0 GiB -> L4)
RAM free: 47.7 GiB | Disk free: 235.6 GiB | CPUs: 12
Auto config -> batch=32, workers=8, cache='disk', TF32=ON
API & Albumentations 70/30 OneOf bbox dry-run passed (FP16 arg: quantize=16).


/usr/local/lib/python3.13/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


Drive image counts -> train: 14691, val: 2150, test: 4200


## 2. Parallel SSD Staging & Vectorized Dataset Check
Stage all 4 CSVs, 6 starter scripts, and 21,041 images to fast local /content/autopilot_cache SSD in parallel and log vectorized integrity stats.

Read-only source preview. Edit the task/section input and rebuild; changing this Markdown does not change execution. The code cell below executes this source verbatim:
```python
CACHE_ROOT = Path('/content/autopilot_cache')
KIT_LOCAL = CACHE_ROOT / 'datasprint_kit'
KIT_LOCAL.mkdir(parents=True, exist_ok=True)

for csv_name in required_csvs:
    rec = autopilot.stage_to_local_cache(PROJECT_ROOT, f'01_data/{csv_name}', cache_root=str(CACHE_ROOT))
    shutil.copy2(rec['local_path'], KIT_LOCAL / csv_name)

for sname in ('benchmark_speed.py', 'check_submission.py', 'inference.py', 'make_kaggle_submission.py', 'score.py', 'train_baseline.py'):
    s_rec = autopilot.stage_to_local_cache(PROJECT_ROOT, f'02_notebooks/starter_kit/{sname}', cache_root=str(CACHE_ROOT))
    shutil.copy2(s_rec['local_path'], KIT_LOCAL / sname)

def _stage_split_fast(split_folder: str):
    src_dir = DRIVE_DATA_DIR / split_folder
    dst_dir = KIT_LOCAL / split_folder
    dst_dir.mkdir(parents=True, exist_ok=True)
    files = [p for p in src_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg'}]
    t0 = time.monotonic()
    def _copy_one(p: Path):
        tgt = dst_dir / p.name
        if not tgt.is_file() or tgt.stat().st_size != p.stat().st_size:
            shutil.copy2(p, tgt)
        return 1
    done = 0
    with ThreadPoolExecutor(max_workers=24) as pool:
        for _ in pool.map(_copy_one, files, chunksize=64):
            done += 1
            if done % 3000 == 0 or done == len(files):
                print(f'  [{split_folder}] staged {done}/{len(files)} ({done / max(0.01, time.monotonic() - t0):.0f} img/s)')
    return len(files)

staged_train = _stage_split_fast('train_images')
staged_val = _stage_split_fast('val_images')
staged_test = _stage_split_fast('test_images')
check_budget()

CLASS_MAP = {0: 'D00_longitudinal_crack', 1: 'D10_transverse_crack', 2: 'D20_alligator_crack', 3: 'D40_pothole'}
train_df = pd.read_csv(KIT_LOCAL / 'train_labels.csv')
val_df = pd.read_csv(KIT_LOCAL / 'val_labels.csv')
val_ids_df = pd.read_csv(KIT_LOCAL / 'val_image_ids.csv')
test_ids_df = pd.read_csv(KIT_LOCAL / 'test_image_ids.csv')

def _vec_split_summary(df: pd.DataFrame, total_imgs: int):
    n_dam = int(df['image_id'].nunique())
    vc = df['class_id'].value_counts().sort_index()
    return {
        'total_images': int(total_imgs),
        'damaged_images': n_dam,
        'background_only_images': int(total_imgs - n_dam),
        'total_boxes': int(len(df)),
        'invalid_boxes': int(((df['x_max'] <= df['x_min']) | (df['y_max'] <= df['y_min'])).sum()),
        'class_counts': {CLASS_MAP[int(k)]: int(v) for k, v in vc.items()},
    }

eda_summary = {
    'train': _vec_split_summary(train_df, staged_train),
    'val': _vec_split_summary(val_df, staged_val),
    'val_image_ids_rows': int(len(val_ids_df)),
    'test_image_ids_rows': int(len(test_ids_df)),
    'staged_test_images': int(staged_test),
}
(RUN_DRIVE_DIR / 'eda_summary.json').write_text(json.dumps(eda_summary, indent=2), encoding='utf-8')
TASK_ARTIFACTS['eda_summary'] = f'03_results/runs/{RUN_ID}/eda_summary.json'
print(json.dumps(eda_summary, indent=2))
```


In [6]:
_execute_stage(1, "CACHE_ROOT = Path('/content/autopilot_cache')\nKIT_LOCAL = CACHE_ROOT / 'datasprint_kit'\nKIT_LOCAL.mkdir(parents=True, exist_ok=True)\n\nfor csv_name in required_csvs:\n    rec = autopilot.stage_to_local_cache(PROJECT_ROOT, f'01_data/{csv_name}', cache_root=str(CACHE_ROOT))\n    shutil.copy2(rec['local_path'], KIT_LOCAL / csv_name)\n\nfor sname in ('benchmark_speed.py', 'check_submission.py', 'inference.py', 'make_kaggle_submission.py', 'score.py', 'train_baseline.py'):\n    s_rec = autopilot.stage_to_local_cache(PROJECT_ROOT, f'02_notebooks/starter_kit/{sname}', cache_root=str(CACHE_ROOT))\n    shutil.copy2(s_rec['local_path'], KIT_LOCAL / sname)\n\ndef _stage_split_fast(split_folder: str):\n    src_dir = DRIVE_DATA_DIR / split_folder\n    dst_dir = KIT_LOCAL / split_folder\n    dst_dir.mkdir(parents=True, exist_ok=True)\n    files = [p for p in src_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg'}]\n    t0 = time.monotonic()\n    def _copy_one(p: Path):\n        tgt = dst_dir / p.name\n        if not tgt.is_file() or tgt.stat().st_size != p.stat().st_size:\n            shutil.copy2(p, tgt)\n        return 1\n    done = 0\n    with ThreadPoolExecutor(max_workers=24) as pool:\n        for _ in pool.map(_copy_one, files, chunksize=64):\n            done += 1\n            if done % 3000 == 0 or done == len(files):\n                print(f'  [{split_folder}] staged {done}/{len(files)} ({done / max(0.01, time.monotonic() - t0):.0f} img/s)')\n    return len(files)\n\nstaged_train = _stage_split_fast('train_images')\nstaged_val = _stage_split_fast('val_images')\nstaged_test = _stage_split_fast('test_images')\ncheck_budget()\n\nCLASS_MAP = {0: 'D00_longitudinal_crack', 1: 'D10_transverse_crack', 2: 'D20_alligator_crack', 3: 'D40_pothole'}\ntrain_df = pd.read_csv(KIT_LOCAL / 'train_labels.csv')\nval_df = pd.read_csv(KIT_LOCAL / 'val_labels.csv')\nval_ids_df = pd.read_csv(KIT_LOCAL / 'val_image_ids.csv')\ntest_ids_df = pd.read_csv(KIT_LOCAL / 'test_image_ids.csv')\n\ndef _vec_split_summary(df: pd.DataFrame, total_imgs: int):\n    n_dam = int(df['image_id'].nunique())\n    vc = df['class_id'].value_counts().sort_index()\n    return {\n        'total_images': int(total_imgs),\n        'damaged_images': n_dam,\n        'background_only_images': int(total_imgs - n_dam),\n        'total_boxes': int(len(df)),\n        'invalid_boxes': int(((df['x_max'] <= df['x_min']) | (df['y_max'] <= df['y_min'])).sum()),\n        'class_counts': {CLASS_MAP[int(k)]: int(v) for k, v in vc.items()},\n    }\n\neda_summary = {\n    'train': _vec_split_summary(train_df, staged_train),\n    'val': _vec_split_summary(val_df, staged_val),\n    'val_image_ids_rows': int(len(val_ids_df)),\n    'test_image_ids_rows': int(len(test_ids_df)),\n    'staged_test_images': int(staged_test),\n}\n(RUN_DRIVE_DIR / 'eda_summary.json').write_text(json.dumps(eda_summary, indent=2), encoding='utf-8')\nTASK_ARTIFACTS['eda_summary'] = f'03_results/runs/{RUN_ID}/eda_summary.json'\nprint(json.dumps(eda_summary, indent=2))")


  [train_images] staged 3000/14691 (8 img/s)
  [train_images] staged 6000/14691 (16 img/s)
  [train_images] staged 9000/14691 (23 img/s)
  [train_images] staged 12000/14691 (31 img/s)
  [train_images] staged 14691/14691 (37 img/s)
  [val_images] staged 2150/2150 (32 img/s)
  [test_images] staged 3000/4200 (25 img/s)
  [test_images] staged 4200/4200 (34 img/s)
{
  "train": {
    "total_images": 14691,
    "damaged_images": 8523,
    "background_only_images": 6168,
    "total_boxes": 17517,
    "invalid_boxes": 0,
    "class_counts": {
      "D00_longitudinal_crack": 4600,
      "D10_transverse_crack": 3132,
      "D20_alligator_crack": 5864,
      "D40_pothole": 3921
    }
  },
  "val": {
    "total_images": 2150,
    "damaged_images": 1254,
    "background_only_images": 896,
    "total_boxes": 2561,
    "invalid_boxes": 0,
    "class_counts": {
      "D00_longitudinal_crack": 695,
      "D10_transverse_crack": 454,
      "D20_alligator_crack": 824,
      "D40_pothole": 588
    }
  },
 

## 3. Vectorized YOLO Labels & EDA-Matched 4-Bucket Degraded Val Set
Compute normalized YOLO coordinates with vectorized NumPy across all 20,078 boxes and build a deterministic 4-bucket degraded validation set matching Plot 05 & Plot 07 test tails.

Read-only source preview. Edit the task/section input and rebuild; changing this Markdown does not change execution. The code cell below executes this source verbatim:
```python
WORK_ROOT = CACHE_ROOT / 'yolo_work' / RUN_ID
YOLO_DATA = WORK_ROOT / 'yolo_data'
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
YOLO_DATA.mkdir(parents=True, exist_ok=True)

def _build_yolo_split_vectorized(split_name: str, lbl_df: pd.DataFrame) -> int:
    img_dir = KIT_LOCAL / f'{split_name}_images'
    out_img = YOLO_DATA / 'images' / split_name
    out_lbl = YOLO_DATA / 'labels' / split_name
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    img_paths = sorted(img_dir.glob('*.jpg'))

    def _read_size(p: Path):
        with Image.open(p) as im:
            return p.stem, im.size[0], im.size[1], p

    with ThreadPoolExecutor(max_workers=16) as pool:
        meta_rows = list(pool.map(_read_size, img_paths, chunksize=64))
    dims_df = pd.DataFrame(meta_rows, columns=['image_id', 'w', 'h', 'path'])

    # Vectorized YOLO normalization across the entire dataframe
    merged = lbl_df.merge(dims_df[['image_id', 'w', 'h']], on='image_id', how='inner')
    xc = np.clip((merged['x_min'].to_numpy(np.float64) + merged['x_max'].to_numpy(np.float64)) * 0.5 / merged['w'].to_numpy(np.float64), 0.0, 1.0)
    yc = np.clip((merged['y_min'].to_numpy(np.float64) + merged['y_max'].to_numpy(np.float64)) * 0.5 / merged['h'].to_numpy(np.float64), 0.0, 1.0)
    bw = np.clip((merged['x_max'].to_numpy(np.float64) - merged['x_min'].to_numpy(np.float64)) / merged['w'].to_numpy(np.float64), 1e-6, 1.0)
    bh = np.clip((merged['y_max'].to_numpy(np.float64) - merged['y_min'].to_numpy(np.float64)) / merged['h'].to_numpy(np.float64), 1e-6, 1.0)
    cids = merged['class_id'].to_numpy(np.int32)
    lines_arr = [f'{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}' for c, x, y, w, h in zip(cids, xc, yc, bw, bh)]
    merged['yolo_line'] = lines_arr
    grouped_txt = merged.groupby('image_id')['yolo_line'].apply(lambda s: '\n'.join(s) + '\n').to_dict()

    def _write_one(row):
        iid, _, _, src_p = row
        dst_i = out_img / src_p.name
        if not dst_i.exists():
            os.symlink(src_p.resolve(), dst_i)
        (out_lbl / f'{iid}.txt').write_text(grouped_txt.get(iid, ''), encoding='utf-8')
        return 1

    with ThreadPoolExecutor(max_workers=16) as pool:
        list(pool.map(_write_one, meta_rows, chunksize=64))
    return len(img_paths)

n_yolo_train = _build_yolo_split_vectorized('train', train_df)
n_yolo_val = _build_yolo_split_vectorized('val', val_df)
DATA_YAML = WORK_ROOT / 'data.yaml'
DATA_YAML.write_text(
    f'path: {YOLO_DATA}\ntrain: images/train\nval: images/val\nnames:\n'
    + ''.join(f'  {k}: {v}\n' for k, v in CLASS_MAP.items()),
    encoding='utf-8',
)
print(f'Vectorized YOLO dataset ready -> train: {n_yolo_train}, val: {n_yolo_val}')

# 4-bucket deterministic degraded val set matching Plot 05 & Plot 07 exact test corruptions
VAL_DEGRADED_DIR = WORK_ROOT / 'val_images_degraded'
VAL_DEGRADED_DIR.mkdir(parents=True, exist_ok=True)
val_jpgs = sorted((KIT_LOCAL / 'val_images').glob('*.jpg'))

_STRESS_BUCKETS = [
    lambda s: A.Compose([A.ImageCompression(compression_type='jpeg', quality_range=(8, 18), p=1.0)], seed=s),
    lambda s: A.Compose([A.RandomBrightnessContrast(brightness_limit=(-0.72, -0.48), contrast_limit=(-0.30, 0.0), p=1.0)], seed=s),
    lambda s: A.Compose([A.OneOf([A.GaussianBlur(blur_limit=(9, 13), p=1.0), A.MotionBlur(blur_limit=(9, 13), p=1.0)], p=1.0)], seed=s),
    lambda s: A.Compose([A.GaussNoise(std_range=(0.06, 0.13), p=1.0)], seed=s),
]

def _degrade_one(idx_path):
    idx, p = idx_path
    pipe = _STRESS_BUCKETS[idx % 4](SEED + idx)
    arr = np.asarray(Image.open(p).convert('RGB'))
    deg = pipe(image=arr)['image']
    Image.fromarray(deg).save(VAL_DEGRADED_DIR / p.name, format='JPEG', quality=88)
    return 1

with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(_degrade_one, enumerate(val_jpgs), chunksize=32))
print(f'Built 4-bucket EDA-calibrated degraded val set: {len(val_jpgs)} images in {VAL_DEGRADED_DIR}')
check_budget()
```


In [7]:
_execute_stage(2, "WORK_ROOT = CACHE_ROOT / 'yolo_work' / RUN_ID\nYOLO_DATA = WORK_ROOT / 'yolo_data'\nif WORK_ROOT.exists():\n    shutil.rmtree(WORK_ROOT)\nYOLO_DATA.mkdir(parents=True, exist_ok=True)\n\ndef _build_yolo_split_vectorized(split_name: str, lbl_df: pd.DataFrame) -> int:\n    img_dir = KIT_LOCAL / f'{split_name}_images'\n    out_img = YOLO_DATA / 'images' / split_name\n    out_lbl = YOLO_DATA / 'labels' / split_name\n    out_img.mkdir(parents=True, exist_ok=True)\n    out_lbl.mkdir(parents=True, exist_ok=True)\n    img_paths = sorted(img_dir.glob('*.jpg'))\n\n    def _read_size(p: Path):\n        with Image.open(p) as im:\n            return p.stem, im.size[0], im.size[1], p\n\n    with ThreadPoolExecutor(max_workers=16) as pool:\n        meta_rows = list(pool.map(_read_size, img_paths, chunksize=64))\n    dims_df = pd.DataFrame(meta_rows, columns=['image_id', 'w', 'h', 'path'])\n\n    # Vectorized YOLO normalization across the entire dataframe\n    merged = lbl_df.merge(dims_df[['image_id', 'w', 'h']], on='image_id', how='inner')\n    xc = np.clip((merged['x_min'].to_numpy(np.float64) + merged['x_max'].to_numpy(np.float64)) * 0.5 / merged['w'].to_numpy(np.float64), 0.0, 1.0)\n    yc = np.clip((merged['y_min'].to_numpy(np.float64) + merged['y_max'].to_numpy(np.float64)) * 0.5 / merged['h'].to_numpy(np.float64), 0.0, 1.0)\n    bw = np.clip((merged['x_max'].to_numpy(np.float64) - merged['x_min'].to_numpy(np.float64)) / merged['w'].to_numpy(np.float64), 1e-6, 1.0)\n    bh = np.clip((merged['y_max'].to_numpy(np.float64) - merged['y_min'].to_numpy(np.float64)) / merged['h'].to_numpy(np.float64), 1e-6, 1.0)\n    cids = merged['class_id'].to_numpy(np.int32)\n    lines_arr = [f'{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}' for c, x, y, w, h in zip(cids, xc, yc, bw, bh)]\n    merged['yolo_line'] = lines_arr\n    grouped_txt = merged.groupby('image_id')['yolo_line'].apply(lambda s: '\\n'.join(s) + '\\n').to_dict()\n\n    def _write_one(row):\n        iid, _, _, src_p = row\n        dst_i = out_img / src_p.name\n        if not dst_i.exists():\n            os.symlink(src_p.resolve(), dst_i)\n        (out_lbl / f'{iid}.txt').write_text(grouped_txt.get(iid, ''), encoding='utf-8')\n        return 1\n\n    with ThreadPoolExecutor(max_workers=16) as pool:\n        list(pool.map(_write_one, meta_rows, chunksize=64))\n    return len(img_paths)\n\nn_yolo_train = _build_yolo_split_vectorized('train', train_df)\nn_yolo_val = _build_yolo_split_vectorized('val', val_df)\nDATA_YAML = WORK_ROOT / 'data.yaml'\nDATA_YAML.write_text(\n    f'path: {YOLO_DATA}\\ntrain: images/train\\nval: images/val\\nnames:\\n'\n    + ''.join(f'  {k}: {v}\\n' for k, v in CLASS_MAP.items()),\n    encoding='utf-8',\n)\nprint(f'Vectorized YOLO dataset ready -> train: {n_yolo_train}, val: {n_yolo_val}')\n\n# 4-bucket deterministic degraded val set matching Plot 05 & Plot 07 exact test corruptions\nVAL_DEGRADED_DIR = WORK_ROOT / 'val_images_degraded'\nVAL_DEGRADED_DIR.mkdir(parents=True, exist_ok=True)\nval_jpgs = sorted((KIT_LOCAL / 'val_images').glob('*.jpg'))\n\n_STRESS_BUCKETS = [\n    lambda s: A.Compose([A.ImageCompression(compression_type='jpeg', quality_range=(8, 18), p=1.0)], seed=s),\n    lambda s: A.Compose([A.RandomBrightnessContrast(brightness_limit=(-0.72, -0.48), contrast_limit=(-0.30, 0.0), p=1.0)], seed=s),\n    lambda s: A.Compose([A.OneOf([A.GaussianBlur(blur_limit=(9, 13), p=1.0), A.MotionBlur(blur_limit=(9, 13), p=1.0)], p=1.0)], seed=s),\n    lambda s: A.Compose([A.GaussNoise(std_range=(0.06, 0.13), p=1.0)], seed=s),\n]\n\ndef _degrade_one(idx_path):\n    idx, p = idx_path\n    pipe = _STRESS_BUCKETS[idx % 4](SEED + idx)\n    arr = np.asarray(Image.open(p).convert('RGB'))\n    deg = pipe(image=arr)['image']\n    Image.fromarray(deg).save(VAL_DEGRADED_DIR / p.name, format='JPEG', quality=88)\n    return 1\n\nwith ThreadPoolExecutor(max_workers=8) as pool:\n    list(pool.map(_degrade_one, enumerate(val_jpgs), chunksize=32))\nprint(f'Built 4-bucket EDA-calibrated degraded val set: {len(val_jpgs)} images in {VAL_DEGRADED_DIR}')\ncheck_budget()")


Vectorized YOLO dataset ready -> train: 14691, val: 2150
Built 4-bucket EDA-calibrated degraded val set: 2150 images in /content/autopilot_cache/yolo_work/001798f3-a360-479d-a2bc-7d389ccf3203/val_images_degraded


## 4. Train yolo26m.pt (70% Clean / 30% OneOf Robustness, 768px, 50 Epochs)
Train COCO yolo26m.pt on all 14,691 images using TF32/AMP, auto-scaled batch & cache, EDA-locked geometry (no 90deg rotation or flipud), close_mosaic=10, and 70% clean / 30% single-degradation OneOf.

Read-only source preview. Edit the task/section input and rebuild; changing this Markdown does not change execution. The code cell below executes this source verbatim:
```python
model_weights = str(EXP_CFG.get('model_weights', 'yolo26m.pt'))
fallback_weights = str(EXP_CFG.get('fallback_weights', 'yolo26s.pt'))
epochs = 1 if EXP_CFG.get('smoke_only') else int(EXP_CFG.get('epochs', 50))
imgsz = int(EXP_CFG.get('imgsz', 768))
close_mosaic = 0 if EXP_CFG.get('smoke_only') else int(EXP_CFG.get('close_mosaic', 10))
patience = int(EXP_CFG.get('patience', 15))
rob_p = float(EXP_CFG.get('robustness_aug_p', 0.30))

try:
    detector = YOLO(model_weights)
    ACTIVE_WEIGHTS_NAME = model_weights
except Exception as exc:
    print(f'Falling back from {model_weights} to {fallback_weights}: {exc}')
    detector = YOLO(fallback_weights)
    ACTIVE_WEIGHTS_NAME = fallback_weights

# 70% CLEAN images (85% normal LB accuracy) + 30% ONE EDA-calibrated degradation
custom_transforms = [
    A.OneOf([
        A.ImageCompression(compression_type='jpeg', quality_range=(6, 22), p=1.0),
        A.Compose([
            A.RandomBrightnessContrast(brightness_limit=(-0.75, -0.45), contrast_limit=(-0.35, 0.0), p=1.0),
            A.RandomGamma(gamma_limit=(140, 220), p=0.5),
        ], p=1.0),
        A.OneOf([
            A.GaussianBlur(blur_limit=(7, 15), p=1.0),
            A.MotionBlur(blur_limit=(7, 15), p=1.0),
        ], p=1.0),
        A.GaussNoise(std_range=(0.05, 0.15), p=1.0),
    ], p=rob_p),
]

detector.train(
    data=str(DATA_YAML),
    epochs=epochs,
    imgsz=imgsz,
    batch=EFFECTIVE_BATCH,
    optimizer=str(EXP_CFG.get('optimizer', 'auto')),
    cos_lr=bool(EXP_CFG.get('cos_lr', True)),
    close_mosaic=close_mosaic,
    mosaic=1.0,
    scale=0.5,
    translate=0.1,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.35,
    erasing=0.15,
    patience=patience,
    seed=SEED,
    deterministic=False,
    amp=True,
    device=0,
    workers=DATALOADER_WORKERS,
    cache=CACHE_MODE,
    project=str(WORK_ROOT / 'runs'),
    name='train_full_v3',
    exist_ok=True,
    plots=False,
    augmentations=custom_transforms,
)
check_budget()

best_pt_src = Path(detector.trainer.best)
if not best_pt_src.is_file():
    raise FileNotFoundError(f'Missing trained best weights at {best_pt_src}')

REPO_DIR = WORK_ROOT / 'submission_repo'
(REPO_DIR / 'weights').mkdir(parents=True, exist_ok=True)
shutil.copy2(best_pt_src, REPO_DIR / 'weights' / 'best.pt')

DRIVE_WEIGHTS_DIR = RUN_DRIVE_DIR / 'weights'
DRIVE_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_pt_src, DRIVE_WEIGHTS_DIR / 'best.pt')
checkpoint_files = [f'03_results/runs/{RUN_ID}/weights/best.pt']
for fn, src_p in [('results.csv', best_pt_src.parent.parent / 'results.csv'), ('last.pt', best_pt_src.parent / 'last.pt')]:
    if src_p.is_file():
        shutil.copy2(src_p, DRIVE_WEIGHTS_DIR / fn)
        checkpoint_files.append(f'03_results/runs/{RUN_ID}/weights/{fn}')

manifest_path = autopilot.save_checkpoint_manifest(PROJECT_ROOT, RUN_ID, 'best_model', CHECKPOINT_CONFIG, checkpoint_files)
TASK_ARTIFACTS['best_weights'] = f'03_results/runs/{RUN_ID}/weights/best.pt'
TASK_ARTIFACTS['checkpoint_manifest'] = str(manifest_path.relative_to(PROJECT_ROOT))
TASK_ARTIFACTS['pretrained_weights'] = ACTIVE_WEIGHTS_NAME
TASK_ARTIFACTS['gpu_tier'] = GPU_TIER
TASK_ARTIFACTS['cache_mode'] = str(CACHE_MODE)
print('Saved checkpoint manifest:', manifest_path)
```


In [8]:
_execute_stage(3, "model_weights = str(EXP_CFG.get('model_weights', 'yolo26m.pt'))\nfallback_weights = str(EXP_CFG.get('fallback_weights', 'yolo26s.pt'))\nepochs = 1 if EXP_CFG.get('smoke_only') else int(EXP_CFG.get('epochs', 50))\nimgsz = int(EXP_CFG.get('imgsz', 768))\nclose_mosaic = 0 if EXP_CFG.get('smoke_only') else int(EXP_CFG.get('close_mosaic', 10))\npatience = int(EXP_CFG.get('patience', 15))\nrob_p = float(EXP_CFG.get('robustness_aug_p', 0.30))\n\ntry:\n    detector = YOLO(model_weights)\n    ACTIVE_WEIGHTS_NAME = model_weights\nexcept Exception as exc:\n    print(f'Falling back from {model_weights} to {fallback_weights}: {exc}')\n    detector = YOLO(fallback_weights)\n    ACTIVE_WEIGHTS_NAME = fallback_weights\n\n# 70% CLEAN images (85% normal LB accuracy) + 30% ONE EDA-calibrated degradation\ncustom_transforms = [\n    A.OneOf([\n        A.ImageCompression(compression_type='jpeg', quality_range=(6, 22), p=1.0),\n        A.Compose([\n            A.RandomBrightnessContrast(brightness_limit=(-0.75, -0.45), contrast_limit=(-0.35, 0.0), p=1.0),\n            A.RandomGamma(gamma_limit=(140, 220), p=0.5),\n        ], p=1.0),\n        A.OneOf([\n            A.GaussianBlur(blur_limit=(7, 15), p=1.0),\n            A.MotionBlur(blur_limit=(7, 15), p=1.0),\n        ], p=1.0),\n        A.GaussNoise(std_range=(0.05, 0.15), p=1.0),\n    ], p=rob_p),\n]\n\ndetector.train(\n    data=str(DATA_YAML),\n    epochs=epochs,\n    imgsz=imgsz,\n    batch=EFFECTIVE_BATCH,\n    optimizer=str(EXP_CFG.get('optimizer', 'auto')),\n    cos_lr=bool(EXP_CFG.get('cos_lr', True)),\n    close_mosaic=close_mosaic,\n    mosaic=1.0,\n    scale=0.5,\n    translate=0.1,\n    degrees=0.0,\n    shear=0.0,\n    perspective=0.0,\n    fliplr=0.5,\n    flipud=0.0,\n    hsv_h=0.015,\n    hsv_s=0.5,\n    hsv_v=0.35,\n    erasing=0.15,\n    patience=patience,\n    seed=SEED,\n    deterministic=False,\n    amp=True,\n    device=0,\n    workers=DATALOADER_WORKERS,\n    cache=CACHE_MODE,\n    project=str(WORK_ROOT / 'runs'),\n    name='train_full_v3',\n    exist_ok=True,\n    plots=False,\n    augmentations=custom_transforms,\n)\ncheck_budget()\n\nbest_pt_src = Path(detector.trainer.best)\nif not best_pt_src.is_file():\n    raise FileNotFoundError(f'Missing trained best weights at {best_pt_src}')\n\nREPO_DIR = WORK_ROOT / 'submission_repo'\n(REPO_DIR / 'weights').mkdir(parents=True, exist_ok=True)\nshutil.copy2(best_pt_src, REPO_DIR / 'weights' / 'best.pt')\n\nDRIVE_WEIGHTS_DIR = RUN_DRIVE_DIR / 'weights'\nDRIVE_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)\nshutil.copy2(best_pt_src, DRIVE_WEIGHTS_DIR / 'best.pt')\ncheckpoint_files = [f'03_results/runs/{RUN_ID}/weights/best.pt']\nfor fn, src_p in [('results.csv', best_pt_src.parent.parent / 'results.csv'), ('last.pt', best_pt_src.parent / 'last.pt')]:\n    if src_p.is_file():\n        shutil.copy2(src_p, DRIVE_WEIGHTS_DIR / fn)\n        checkpoint_files.append(f'03_results/runs/{RUN_ID}/weights/{fn}')\n\nmanifest_path = autopilot.save_checkpoint_manifest(PROJECT_ROOT, RUN_ID, 'best_model', CHECKPOINT_CONFIG, checkpoint_files)\nTASK_ARTIFACTS['best_weights'] = f'03_results/runs/{RUN_ID}/weights/best.pt'\nTASK_ARTIFACTS['checkpoint_manifest'] = str(manifest_path.relative_to(PROJECT_ROOT))\nTASK_ARTIFACTS['pretrained_weights'] = ACTIVE_WEIGHTS_NAME\nTASK_ARTIFACTS['gpu_tier'] = GPU_TIER\nTASK_ARTIFACTS['cache_mode'] = str(CACHE_MODE)\nprint('Saved checkpoint manifest:', manifest_path)")


Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[{'__version__': '2.0.8', 'transform': {'__class_fullname__': 'OneOf', 'p': 0.3, 'transforms': [{'__class_fullname__': 'ImageCompression', 'p': 1.0, 'compression_type': 'jpeg', 'quality_range': (6, 22)}, {'__class_fullname__': 'Compose', 'p': 1.0, 'transforms': [{'__class_fullname__': 'RandomBrightnessContrast', 'p': 1.0, 'brightness_by_max': True, 'brightness_limit': (-0.75, -0.45), 'contrast_limit': (-0.35, 0.0), 'ensure_safe_range': False}, {'__class_fullname__': 'RandomGamma', 'p': 0.5, 'gamma_limit': (140.0, 220.0)}], 'bbox_params': None, 'keypoint_params': None, 'additional_targets': {}, 'is_check_shapes': True}, {'__class_fullname__': 'OneOf', 'p': 1.0, 'transforms': [{'__class_fullname__': 'GaussianBlur', 'p': 1.0, 'blur_limit': (7, 15), 'sigma_limit': (0.5, 3.0)}, {'__class_fullname__': 'MotionBlur', 'p': 1.0

RuntimeError: Runtime budget reached; save a checkpoint and stop

In [9]:
import os, shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

RUN_ID = "001798f3-a360-479d-a2bc-7d389ccf3203"
DRIVE_WEIGHTS = Path("/content/drive/MyDrive/DataSprint_2.0_Road_Damage/03_results/runs") / RUN_ID / "weights"
DRIVE_WEIGHTS.mkdir(parents=True, exist_ok=True)

# Locate best.pt anywhere under /content
direct = Path(f"/content/autopilot_cache/yolo_work/{RUN_ID}/runs/train_full_v3/weights/best.pt")
cands = [direct] if direct.is_file() else list(Path("/content").rglob("best.pt"))
print("Found best.pt candidates:", [str(p) for p in cands])

if not cands:
    raise FileNotFoundError("No best.pt found under /content — Colab may have recycled the VM.")

best_pt = cands[0]
for fn, src in [
    ("best.pt", best_pt),
    ("last.pt", best_pt.parent / "last.pt"),
    ("results.csv", best_pt.parent.parent / "results.csv"),
]:
    if src.is_file():
        dst = DRIVE_WEIGHTS / fn
        shutil.copy2(src, dst)
        print(f"SAVED -> {dst} ({dst.stat().st_size / 1e6:.2f} MB)")

os.sync()
print("DONE! Safely backed up to Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found best.pt candidates: ['/content/autopilot_cache/yolo_work/001798f3-a360-479d-a2bc-7d389ccf3203/runs/train_full_v3/weights/best.pt']
SAVED -> /content/drive/MyDrive/DataSprint_2.0_Road_Damage/03_results/runs/001798f3-a360-479d-a2bc-7d389ccf3203/weights/best.pt (44.06 MB)
SAVED -> /content/drive/MyDrive/DataSprint_2.0_Road_Damage/03_results/runs/001798f3-a360-479d-a2bc-7d389ccf3203/weights/last.pt (44.06 MB)
SAVED -> /content/drive/MyDrive/DataSprint_2.0_Road_Damage/03_results/runs/001798f3-a360-479d-a2bc-7d389ccf3203/weights/results.csv (0.01 MB)
DONE! Safely backed up to Google Drive.


## 5. A/B Test Conditional Low-Light Rescue & Score Clean + Degraded Val
Write Variant A (FP16 raw) and Variant B (FP16 + <1ms LUT brightening when strided luma < 60), benchmark both on clean + 4-bucket degraded val, and auto-promote the higher Final Score winner.

Read-only source preview. Edit the task/section input and rebuild; changing this Markdown does not change execution. The code cell below executes this source verbatim:
```python
conf_floor = float(EXP_CFG.get('conf_floor', 0.001))
iou_nms = float(EXP_CFG.get('iou_nms', 0.6))
max_det = int(EXP_CFG.get('max_det', 100))
use_fp16 = bool(EXP_CFG.get('use_fp16_inference', True))
luma_thresh = float(EXP_CFG.get('lowlight_luma_thresh', 60.0))
target_luma = float(EXP_CFG.get('lowlight_target_luma', 115.0))

def _make_inference_py(enable_lowlight_rescue: bool) -> str:
    return f'''import math
import os
import numpy as np

_HERE = os.path.dirname(os.path.abspath(__file__))
WEIGHTS = os.path.join(_HERE, "weights", "best.pt")
IMGSZ = {imgsz}
USE_FP16 = {use_fp16!r}
ENABLE_LOWLIGHT_RESCUE = {enable_lowlight_rescue!r}
LUMA_THRESH = {luma_thresh}
TARGET_LUMA = {target_luma}

def _build_lut_table():
    x = np.arange(256, dtype=np.float32) / 255.0
    tgt = math.log(TARGET_LUMA / 255.0)
    luts = {{}}
    for lum in range(int(LUMA_THRESH)):
        gamma = max(0.35, min(0.85, tgt / math.log(max(lum, 8.0) / 255.0)))
        luts[lum] = np.clip(np.round(255.0 * (x ** gamma)), 0, 255).astype(np.uint8)
    return luts

class _Model:
    def __init__(self, yolo, device, fp16_kwargs, luts):
        self.yolo = yolo
        self.device = device
        self.fp16_kwargs = fp16_kwargs
        self.luts = luts

def load(device="cuda"):
    from ultralytics import YOLO
    from ultralytics.cfg import DEFAULT_CFG_DICT
    fp16_kwargs = {{}}
    if USE_FP16 and str(device).startswith("cuda"):
        if "quantize" in DEFAULT_CFG_DICT:
            fp16_kwargs["quantize"] = 16
        else:
            fp16_kwargs["half"] = True
    luts = _build_lut_table() if ENABLE_LOWLIGHT_RESCUE else None
    return _Model(YOLO(WEIGHTS), device, fp16_kwargs, luts)

def predict(model, image):
    inp = image
    if model.luts is not None:
        mean_lum = float(image[::4, ::4].mean())
        if mean_lum < LUMA_THRESH:
            inp = model.luts[max(0, min(int(LUMA_THRESH) - 1, int(mean_lum)))][image]
    bgr = np.ascontiguousarray(inp[:, :, ::-1])
    r = model.yolo.predict(
        bgr,
        imgsz=IMGSZ,
        conf={conf_floor},
        iou={iou_nms},
        max_det={max_det},
        agnostic_nms=False,
        device=model.device,
        verbose=False,
        **model.fp16_kwargs,
    )[0]
    if r.boxes is None or len(r.boxes) == 0:
        return []
    xyxy = r.boxes.xyxy.cpu().numpy()
    conf = r.boxes.conf.cpu().numpy()
    cls = r.boxes.cls.cpu().numpy().astype(int)
    return [(int(c), float(s), float(b[0]), float(b[1]), float(b[2]), float(b[3]))
            for c, s, b in zip(cls, conf, xyxy)]
'''

def _eval_variant(tag: str, enable_rescue: bool):
    code_str = _make_inference_py(enable_rescue)
    (REPO_DIR / 'inference.py').write_text(code_str, encoding='utf-8')
    v_pred = WORK_ROOT / f'{tag}_val_pred.csv'
    v_spd = WORK_ROOT / f'{tag}_val_speed.json'
    v_scr = WORK_ROOT / f'{tag}_val_score.json'
    s_pred = WORK_ROOT / f'{tag}_stress_pred.csv'
    s_spd = WORK_ROOT / f'{tag}_stress_speed.json'
    s_scr = WORK_ROOT / f'{tag}_stress_score.json'
    subprocess.run([sys.executable, str(KIT_LOCAL / 'benchmark_speed.py'), '--repo', str(REPO_DIR), '--images', str(KIT_LOCAL / 'val_images'), '--out-csv', str(v_pred), '--out-json', str(v_spd), '--runs', '2', '--repeat-images', '300'], check=True)
    subprocess.run([sys.executable, str(KIT_LOCAL / 'score.py'), '--pred', str(v_pred), '--gt', str(KIT_LOCAL / 'val_labels.csv'), '--image-ids', str(KIT_LOCAL / 'val_image_ids.csv'), '--speed-json', str(v_spd), '--json-out', str(v_scr)], check=True)
    subprocess.run([sys.executable, str(KIT_LOCAL / 'benchmark_speed.py'), '--repo', str(REPO_DIR), '--images', str(VAL_DEGRADED_DIR), '--out-csv', str(s_pred), '--out-json', str(s_spd), '--runs', '1', '--warmup', '5'], check=True)
    subprocess.run([sys.executable, str(KIT_LOCAL / 'score.py'), '--pred', str(s_pred), '--gt', str(KIT_LOCAL / 'val_labels.csv'), '--image-ids', str(KIT_LOCAL / 'val_image_ids.csv'), '--json-out', str(s_scr)], check=True)
    spd = json.loads(v_spd.read_text(encoding='utf-8'))
    scr = json.loads(v_scr.read_text(encoding='utf-8'))
    st_scr = json.loads(s_scr.read_text(encoding='utf-8'))
    m = scr['main']
    rob = float(st_scr['main']['mAP50'])
    spd_sc = float(spd['speed_score'])
    final_sc = 100.0 * (0.30 * float(m['mAP50']) + 0.25 * float(m['mAP50_95']) + 0.15 * float(m['precision']) + 0.15 * float(m['recall']) + 0.10 * spd_sc + 0.05 * rob)
    return {
        'tag': tag, 'enable_lowlight_rescue': enable_rescue, 'code': code_str,
        'val_speed': spd, 'val_score': scr, 'stress_score': st_scr,
        'final_score_proxy_100': final_sc, 'robustness_mAP50_synth': rob,
        'v_spd_path': v_spd, 'v_scr_path': v_scr, 's_scr_path': s_scr,
    }

res_raw = _eval_variant('raw', False)
res_lut = _eval_variant('lowlight_lut', True)
winner = res_lut if res_lut['final_score_proxy_100'] >= res_raw['final_score_proxy_100'] else res_raw
ab_delta = float(res_lut['final_score_proxy_100'] - res_raw['final_score_proxy_100'])

(REPO_DIR / 'inference.py').write_text(winner['code'], encoding='utf-8')
shutil.copy2(REPO_DIR / 'inference.py', RUN_DRIVE_DIR / 'inference.py')
shutil.copy2(winner['v_spd_path'], RUN_DRIVE_DIR / 'val_speed.json')
shutil.copy2(winner['v_scr_path'], RUN_DRIVE_DIR / 'val_score.json')
shutil.copy2(winner['s_scr_path'], RUN_DRIVE_DIR / 'stress_score.json')

ab_summary = {
    'raw': {'accuracy_score_100': res_raw['val_score']['combined']['accuracy_score'], 'robustness_mAP50': res_raw['robustness_mAP50_synth'], 'median_ms': res_raw['val_speed']['median_ms'], 'final_score_proxy_100': round(res_raw['final_score_proxy_100'], 4)},
    'lowlight_lut': {'accuracy_score_100': res_lut['val_score']['combined']['accuracy_score'], 'robustness_mAP50': res_lut['robustness_mAP50_synth'], 'median_ms': res_lut['val_speed']['median_ms'], 'final_score_proxy_100': round(res_lut['final_score_proxy_100'], 4)},
    'winner_tag': winner['tag'],
    'ab_lowlight_delta_final_pts': round(ab_delta, 4),
}
(RUN_DRIVE_DIR / 'ab_inference_comparison.json').write_text(json.dumps(ab_summary, indent=2), encoding='utf-8')
print('A/B Inference Comparison:', json.dumps(ab_summary, indent=2))

m = winner['val_score']['main']
pc = m['per_class']
telemetry = autopilot.collect_colab_telemetry(time.monotonic() - STARTED, burn_rate_cu_per_hour=CONFIG.get('budget', {}).get('burn_rate_cu_per_hour'))
metrics.update({
    'accuracy_score_100': float(winner['val_score']['combined']['accuracy_score']),
    'mAP50': float(m['mAP50']),
    'mAP50_95': float(m['mAP50_95']),
    'precision': float(m['precision']),
    'recall': float(m['recall']),
    'ap50_D00': float(pc['D00_longitudinal_crack']['AP50']),
    'ap50_D10': float(pc['D10_transverse_crack']['AP50']),
    'ap50_D20': float(pc['D20_alligator_crack']['AP50']),
    'ap50_D40': float(pc['D40_pothole']['AP50']),
    'speed_score_t4': float(winner['val_speed']['speed_score']),
    'median_ms_t4': float(winner['val_speed']['median_ms']),
    'robustness_mAP50_synth': float(winner['robustness_mAP50_synth']),
    'final_score_proxy_100': float(winner['final_score_proxy_100']),
    'ab_lowlight_delta_final_pts': float(ab_delta),
    'winner_uses_lowlight_rescue': 1 if winner['enable_lowlight_rescue'] else 0,
    'train_images_count': int(n_yolo_train),
    'val_images_count': int(n_yolo_val),
    'effective_batch_size': int(EFFECTIVE_BATCH),
})
if 'peak_vram_gib' in telemetry:
    metrics['peak_vram_gib'] = float(telemetry['peak_vram_gib'])
if 'estimated_compute_units' in telemetry:
    metrics['estimated_compute_units'] = float(telemetry['estimated_compute_units'])

TASK_ARTIFACTS.update({
    'val_score_json': f'03_results/runs/{RUN_ID}/val_score.json',
    'val_speed_json': f'03_results/runs/{RUN_ID}/val_speed.json',
    'stress_score_json': f'03_results/runs/{RUN_ID}/stress_score.json',
    'ab_inference_comparison': f'03_results/runs/{RUN_ID}/ab_inference_comparison.json',
    'inference_py': f'03_results/runs/{RUN_ID}/inference.py',
    'winner_inference_variant': winner['tag'],
})
if 'T4' not in GPU_NAME:
    print(f'NOTE: Speed benchmark ran on {GPU_NAME} ({winner["val_speed"]["median_ms"]} ms). Verify <=25ms on a Tesla T4 runtime before final submission.')
print('Validation Summary:', json.dumps(metrics, indent=2))
check_budget()
```


In [ ]:
_execute_stage(4, 'conf_floor = float(EXP_CFG.get(\'conf_floor\', 0.001))\niou_nms = float(EXP_CFG.get(\'iou_nms\', 0.6))\nmax_det = int(EXP_CFG.get(\'max_det\', 100))\nuse_fp16 = bool(EXP_CFG.get(\'use_fp16_inference\', True))\nluma_thresh = float(EXP_CFG.get(\'lowlight_luma_thresh\', 60.0))\ntarget_luma = float(EXP_CFG.get(\'lowlight_target_luma\', 115.0))\n\ndef _make_inference_py(enable_lowlight_rescue: bool) -> str:\n    return f\'\'\'import math\nimport os\nimport numpy as np\n\n_HERE = os.path.dirname(os.path.abspath(__file__))\nWEIGHTS = os.path.join(_HERE, "weights", "best.pt")\nIMGSZ = {imgsz}\nUSE_FP16 = {use_fp16!r}\nENABLE_LOWLIGHT_RESCUE = {enable_lowlight_rescue!r}\nLUMA_THRESH = {luma_thresh}\nTARGET_LUMA = {target_luma}\n\ndef _build_lut_table():\n    x = np.arange(256, dtype=np.float32) / 255.0\n    tgt = math.log(TARGET_LUMA / 255.0)\n    luts = {{}}\n    for lum in range(int(LUMA_THRESH)):\n        gamma = max(0.35, min(0.85, tgt / math.log(max(lum, 8.0) / 255.0)))\n        luts[lum] = np.clip(np.round(255.0 * (x ** gamma)), 0, 255).astype(np.uint8)\n    return luts\n\nclass _Model:\n    def __init__(self, yolo, device, fp16_kwargs, luts):\n        self.yolo = yolo\n        self.device = device\n        self.fp16_kwargs = fp16_kwargs\n        self.luts = luts\n\ndef load(device="cuda"):\n    from ultralytics import YOLO\n    from ultralytics.cfg import DEFAULT_CFG_DICT\n    fp16_kwargs = {{}}\n    if USE_FP16 and str(device).startswith("cuda"):\n        if "quantize" in DEFAULT_CFG_DICT:\n            fp16_kwargs["quantize"] = 16\n        else:\n            fp16_kwargs["half"] = True\n    luts = _build_lut_table() if ENABLE_LOWLIGHT_RESCUE else None\n    return _Model(YOLO(WEIGHTS), device, fp16_kwargs, luts)\n\ndef predict(model, image):\n    inp = image\n    if model.luts is not None:\n        mean_lum = float(image[::4, ::4].mean())\n        if mean_lum < LUMA_THRESH:\n            inp = model.luts[max(0, min(int(LUMA_THRESH) - 1, int(mean_lum)))][image]\n    bgr = np.ascontiguousarray(inp[:, :, ::-1])\n    r = model.yolo.predict(\n        bgr,\n        imgsz=IMGSZ,\n        conf={conf_floor},\n        iou={iou_nms},\n        max_det={max_det},\n        agnostic_nms=False,\n        device=model.device,\n        verbose=False,\n        **model.fp16_kwargs,\n    )[0]\n    if r.boxes is None or len(r.boxes) == 0:\n        return []\n    xyxy = r.boxes.xyxy.cpu().numpy()\n    conf = r.boxes.conf.cpu().numpy()\n    cls = r.boxes.cls.cpu().numpy().astype(int)\n    return [(int(c), float(s), float(b[0]), float(b[1]), float(b[2]), float(b[3]))\n            for c, s, b in zip(cls, conf, xyxy)]\n\'\'\'\n\ndef _eval_variant(tag: str, enable_rescue: bool):\n    code_str = _make_inference_py(enable_rescue)\n    (REPO_DIR / \'inference.py\').write_text(code_str, encoding=\'utf-8\')\n    v_pred = WORK_ROOT / f\'{tag}_val_pred.csv\'\n    v_spd = WORK_ROOT / f\'{tag}_val_speed.json\'\n    v_scr = WORK_ROOT / f\'{tag}_val_score.json\'\n    s_pred = WORK_ROOT / f\'{tag}_stress_pred.csv\'\n    s_spd = WORK_ROOT / f\'{tag}_stress_speed.json\'\n    s_scr = WORK_ROOT / f\'{tag}_stress_score.json\'\n    subprocess.run([sys.executable, str(KIT_LOCAL / \'benchmark_speed.py\'), \'--repo\', str(REPO_DIR), \'--images\', str(KIT_LOCAL / \'val_images\'), \'--out-csv\', str(v_pred), \'--out-json\', str(v_spd), \'--runs\', \'2\', \'--repeat-images\', \'300\'], check=True)\n    subprocess.run([sys.executable, str(KIT_LOCAL / \'score.py\'), \'--pred\', str(v_pred), \'--gt\', str(KIT_LOCAL / \'val_labels.csv\'), \'--image-ids\', str(KIT_LOCAL / \'val_image_ids.csv\'), \'--speed-json\', str(v_spd), \'--json-out\', str(v_scr)], check=True)\n    subprocess.run([sys.executable, str(KIT_LOCAL / \'benchmark_speed.py\'), \'--repo\', str(REPO_DIR), \'--images\', str(VAL_DEGRADED_DIR), \'--out-csv\', str(s_pred), \'--out-json\', str(s_spd), \'--runs\', \'1\', \'--warmup\', \'5\'], check=True)\n    subprocess.run([sys.executable, str(KIT_LOCAL / \'score.py\'), \'--pred\', str(s_pred), \'--gt\', str(KIT_LOCAL / \'val_labels.csv\'), \'--image-ids\', str(KIT_LOCAL / \'val_image_ids.csv\'), \'--json-out\', str(s_scr)], check=True)\n    spd = json.loads(v_spd.read_text(encoding=\'utf-8\'))\n    scr = json.loads(v_scr.read_text(encoding=\'utf-8\'))\n    st_scr = json.loads(s_scr.read_text(encoding=\'utf-8\'))\n    m = scr[\'main\']\n    rob = float(st_scr[\'main\'][\'mAP50\'])\n    spd_sc = float(spd[\'speed_score\'])\n    final_sc = 100.0 * (0.30 * float(m[\'mAP50\']) + 0.25 * float(m[\'mAP50_95\']) + 0.15 * float(m[\'precision\']) + 0.15 * float(m[\'recall\']) + 0.10 * spd_sc + 0.05 * rob)\n    return {\n        \'tag\': tag, \'enable_lowlight_rescue\': enable_rescue, \'code\': code_str,\n        \'val_speed\': spd, \'val_score\': scr, \'stress_score\': st_scr,\n        \'final_score_proxy_100\': final_sc, \'robustness_mAP50_synth\': rob,\n        \'v_spd_path\': v_spd, \'v_scr_path\': v_scr, \'s_scr_path\': s_scr,\n    }\n\nres_raw = _eval_variant(\'raw\', False)\nres_lut = _eval_variant(\'lowlight_lut\', True)\nwinner = res_lut if res_lut[\'final_score_proxy_100\'] >= res_raw[\'final_score_proxy_100\'] else res_raw\nab_delta = float(res_lut[\'final_score_proxy_100\'] - res_raw[\'final_score_proxy_100\'])\n\n(REPO_DIR / \'inference.py\').write_text(winner[\'code\'], encoding=\'utf-8\')\nshutil.copy2(REPO_DIR / \'inference.py\', RUN_DRIVE_DIR / \'inference.py\')\nshutil.copy2(winner[\'v_spd_path\'], RUN_DRIVE_DIR / \'val_speed.json\')\nshutil.copy2(winner[\'v_scr_path\'], RUN_DRIVE_DIR / \'val_score.json\')\nshutil.copy2(winner[\'s_scr_path\'], RUN_DRIVE_DIR / \'stress_score.json\')\n\nab_summary = {\n    \'raw\': {\'accuracy_score_100\': res_raw[\'val_score\'][\'combined\'][\'accuracy_score\'], \'robustness_mAP50\': res_raw[\'robustness_mAP50_synth\'], \'median_ms\': res_raw[\'val_speed\'][\'median_ms\'], \'final_score_proxy_100\': round(res_raw[\'final_score_proxy_100\'], 4)},\n    \'lowlight_lut\': {\'accuracy_score_100\': res_lut[\'val_score\'][\'combined\'][\'accuracy_score\'], \'robustness_mAP50\': res_lut[\'robustness_mAP50_synth\'], \'median_ms\': res_lut[\'val_speed\'][\'median_ms\'], \'final_score_proxy_100\': round(res_lut[\'final_score_proxy_100\'], 4)},\n    \'winner_tag\': winner[\'tag\'],\n    \'ab_lowlight_delta_final_pts\': round(ab_delta, 4),\n}\n(RUN_DRIVE_DIR / \'ab_inference_comparison.json\').write_text(json.dumps(ab_summary, indent=2), encoding=\'utf-8\')\nprint(\'A/B Inference Comparison:\', json.dumps(ab_summary, indent=2))\n\nm = winner[\'val_score\'][\'main\']\npc = m[\'per_class\']\ntelemetry = autopilot.collect_colab_telemetry(time.monotonic() - STARTED, burn_rate_cu_per_hour=CONFIG.get(\'budget\', {}).get(\'burn_rate_cu_per_hour\'))\nmetrics.update({\n    \'accuracy_score_100\': float(winner[\'val_score\'][\'combined\'][\'accuracy_score\']),\n    \'mAP50\': float(m[\'mAP50\']),\n    \'mAP50_95\': float(m[\'mAP50_95\']),\n    \'precision\': float(m[\'precision\']),\n    \'recall\': float(m[\'recall\']),\n    \'ap50_D00\': float(pc[\'D00_longitudinal_crack\'][\'AP50\']),\n    \'ap50_D10\': float(pc[\'D10_transverse_crack\'][\'AP50\']),\n    \'ap50_D20\': float(pc[\'D20_alligator_crack\'][\'AP50\']),\n    \'ap50_D40\': float(pc[\'D40_pothole\'][\'AP50\']),\n    \'speed_score_t4\': float(winner[\'val_speed\'][\'speed_score\']),\n    \'median_ms_t4\': float(winner[\'val_speed\'][\'median_ms\']),\n    \'robustness_mAP50_synth\': float(winner[\'robustness_mAP50_synth\']),\n    \'final_score_proxy_100\': float(winner[\'final_score_proxy_100\']),\n    \'ab_lowlight_delta_final_pts\': float(ab_delta),\n    \'winner_uses_lowlight_rescue\': 1 if winner[\'enable_lowlight_rescue\'] else 0,\n    \'train_images_count\': int(n_yolo_train),\n    \'val_images_count\': int(n_yolo_val),\n    \'effective_batch_size\': int(EFFECTIVE_BATCH),\n})\nif \'peak_vram_gib\' in telemetry:\n    metrics[\'peak_vram_gib\'] = float(telemetry[\'peak_vram_gib\'])\nif \'estimated_compute_units\' in telemetry:\n    metrics[\'estimated_compute_units\'] = float(telemetry[\'estimated_compute_units\'])\n\nTASK_ARTIFACTS.update({\n    \'val_score_json\': f\'03_results/runs/{RUN_ID}/val_score.json\',\n    \'val_speed_json\': f\'03_results/runs/{RUN_ID}/val_speed.json\',\n    \'stress_score_json\': f\'03_results/runs/{RUN_ID}/stress_score.json\',\n    \'ab_inference_comparison\': f\'03_results/runs/{RUN_ID}/ab_inference_comparison.json\',\n    \'inference_py\': f\'03_results/runs/{RUN_ID}/inference.py\',\n    \'winner_inference_variant\': winner[\'tag\'],\n})\nif \'T4\' not in GPU_NAME:\n    print(f\'NOTE: Speed benchmark ran on {GPU_NAME} ({winner["val_speed"]["median_ms"]} ms). Verify <=25ms on a Tesla T4 runtime before final submission.\')\nprint(\'Validation Summary:\', json.dumps(metrics, indent=2))\ncheck_budget()')


## 6. Generate & Validate Kaggle Submission for All 4,200 Test Images
Predict all 4,200 test images using the winning inference.py, convert with make_kaggle_submission.py, verify zero format errors via check_submission.py, and save to 04_submissions/.

Read-only source preview. Edit the task/section input and rebuild; changing this Markdown does not change execution. The code cell below executes this source verbatim:
```python
expected_test_ids = len(test_ids_df)
actual_test_imgs = len(list((KIT_LOCAL / 'test_images').glob('*.jpg')))
if actual_test_imgs < expected_test_ids:
    raise RuntimeError(f'Expected {expected_test_ids} test images, got {actual_test_imgs}')

test_pred_csv = WORK_ROOT / 'test_pred.csv'
test_speed_json = WORK_ROOT / 'test_speed.json'
sub_csv = WORK_ROOT / f'submission_{RUN_ID}.csv'
subprocess.run([sys.executable, str(KIT_LOCAL / 'benchmark_speed.py'), '--repo', str(REPO_DIR), '--images', str(KIT_LOCAL / 'test_images'), '--out-csv', str(test_pred_csv), '--out-json', str(test_speed_json), '--runs', '1', '--warmup', '10'], check=True)
subprocess.run([sys.executable, str(KIT_LOCAL / 'make_kaggle_submission.py'), '--pred', str(test_pred_csv), '--image-ids', str(KIT_LOCAL / 'test_image_ids.csv'), '--out', str(sub_csv)], check=True)
subprocess.run([sys.executable, str(KIT_LOCAL / 'check_submission.py'), '--pred', str(sub_csv), '--image-ids', str(KIT_LOCAL / 'test_image_ids.csv')], check=True)

drive_sub = PROJECT_ROOT / '04_submissions' / f'submission_{RUN_ID}.csv'
shutil.copy2(sub_csv, drive_sub)
shutil.copy2(test_speed_json, RUN_DRIVE_DIR / 'test_speed.json')
TASK_ARTIFACTS['submission_csv'] = f'04_submissions/submission_{RUN_ID}.csv'
TASK_ARTIFACTS['submission_sha256'] = autopilot.file_digest(drive_sub)
TASK_ARTIFACTS['submission_status'] = 'validated_by_check_submission_py'
print('Validated Kaggle submission saved to:', drive_sub)
```


In [10]:
_execute_stage(5, "expected_test_ids = len(test_ids_df)\nactual_test_imgs = len(list((KIT_LOCAL / 'test_images').glob('*.jpg')))\nif actual_test_imgs < expected_test_ids:\n    raise RuntimeError(f'Expected {expected_test_ids} test images, got {actual_test_imgs}')\n\ntest_pred_csv = WORK_ROOT / 'test_pred.csv'\ntest_speed_json = WORK_ROOT / 'test_speed.json'\nsub_csv = WORK_ROOT / f'submission_{RUN_ID}.csv'\nsubprocess.run([sys.executable, str(KIT_LOCAL / 'benchmark_speed.py'), '--repo', str(REPO_DIR), '--images', str(KIT_LOCAL / 'test_images'), '--out-csv', str(test_pred_csv), '--out-json', str(test_speed_json), '--runs', '1', '--warmup', '10'], check=True)\nsubprocess.run([sys.executable, str(KIT_LOCAL / 'make_kaggle_submission.py'), '--pred', str(test_pred_csv), '--image-ids', str(KIT_LOCAL / 'test_image_ids.csv'), '--out', str(sub_csv)], check=True)\nsubprocess.run([sys.executable, str(KIT_LOCAL / 'check_submission.py'), '--pred', str(sub_csv), '--image-ids', str(KIT_LOCAL / 'test_image_ids.csv')], check=True)\n\ndrive_sub = PROJECT_ROOT / '04_submissions' / f'submission_{RUN_ID}.csv'\nshutil.copy2(sub_csv, drive_sub)\nshutil.copy2(test_speed_json, RUN_DRIVE_DIR / 'test_speed.json')\nTASK_ARTIFACTS['submission_csv'] = f'04_submissions/submission_{RUN_ID}.csv'\nTASK_ARTIFACTS['submission_sha256'] = autopilot.file_digest(drive_sub)\nTASK_ARTIFACTS['submission_status'] = 'validated_by_check_submission_py'\nprint('Validated Kaggle submission saved to:', drive_sub)")


RuntimeError: Run task sections once in order; recover or create a new run after failure

## Verify and persist results
This cell records completion only after every section succeeds. If delivery fails, recover the spool rather than repeat the experiment.


In [ ]:
if _STAGE_STATE['failed'] or _STAGE_STATE['finalized'] or _STAGE_STATE['next'] != 6:
    raise RuntimeError('All task sections must succeed before finalization')
try:
    autopilot.validate_metrics(metrics)
    elapsed_seconds = time.monotonic() - STARTED
    limit = CONFIG.get('budget', {}).get('max_hours')
    if limit is not None and elapsed_seconds > limit * 3600:
        raise RuntimeError('Observed runtime exceeded configured limit')
    log_path = autopilot.log_experiment(PROJECT_ROOT, CONFIG.get('experiment_name', 'experiment'), metrics, run_id=RUN_ID, spool_root=SPOOL_ROOT, config={'project_id': CONFIG['project_id'], 'experiment': CONFIG.get('experiment', {}), 'environment': ENVIRONMENT, 'source_fingerprints': CONFIG.get('source_fingerprints', {})}, context=CONFIG.get('context', {}), metric_context=CONFIG.get('metric_context', {}), artifacts={**TASK_ARTIFACTS, 'elapsed_seconds': elapsed_seconds})
    print('Completed run:', RUN_ID, 'Log:', log_path)
except autopilot.LogDeliveryError as error:
    _STAGE_STATE['failed'] = True
    print('Result delivery is pending. Preserve the local spool and use RECOVER_RUN_ID:', RUN_ID)
    raise
except BaseException as error:
    _STAGE_STATE['failed'] = True
    status = 'interrupted' if isinstance(error, KeyboardInterrupt) else 'failed'
    try:
        autopilot.log_experiment(PROJECT_ROOT, CONFIG.get('experiment_name', 'experiment'), {}, run_id=RUN_ID, event_id='failure-' + str(uuid.uuid4()), status=status, spool_root=SPOOL_ROOT, config={'project_id': CONFIG['project_id'], 'experiment': CONFIG.get('experiment', {}), 'environment': ENVIRONMENT, 'source_fingerprints': CONFIG.get('source_fingerprints', {})}, errors=[{'type': type(error).__name__}])
    except Exception:
        print('Could not persist failure event; inspect runtime storage before disconnecting.')
    raise
_STAGE_STATE['finalized'] = True
